# 📗 트랜스포머 인코더: 토큰화부터 감성 분석까지

트랜스포머(transformer)는 모든 토큰이 다른 모든 토큰을 한꺼번에 보는 어텐션 층을 쌓은 모델입니다. 이 교안은 영화 리뷰를 문자 토큰으로 바꾸고 어텐션, 위치 인코딩, Multi-Head Attention, 인코더 블록을 차례로 만들어 긍정, 부정 분류 모델을 학습합니다. 교안 02에서는 같은 인코더를 nn 모듈 없이 직접 구현합니다.

- **인코더(encoder):** 입력을 모델이 계산하기 좋은 숫자 표현(벡터)으로 바꾸는 부분입니다. 인코딩(encoding)은 정보를 다른 형식으로 바꾸는 일입니다.
- **디코더(decoder):** 그 표현을 원하는 출력(문장 등)으로 풀어내는 부분입니다. 디코딩(decoding)은 바꾼 형식을 다시 풀어내는 일입니다.
  - 예: 번역기는 인코더가 한국어 문장을 벡터로 바꾸고, 디코더가 그 벡터를 보고 영어 문장을 한 토큰씩 만듭니다.
  - 이 교안은 인코더를 만들어 리뷰를 벡터로 바꾸고 분류합니다. 디코더는 개념만 다룹니다.
- **논문:** [Vaswani et al., 2017, Attention Is All You Need](https://arxiv.org/abs/1706.03762)

<img src="images/20_lesson01_goals.png" width="1050" alt="제목은 '이번 교안에서 배울 것: 토큰화부터 감성 분석까지'입니다. 구역 6개가 2줄로 놓여 있습니다. 1 트랜스포머 구조는 인코더 상자에서 디코더 상자로 가는 화살표와 '이 교안에서 만듦', '개념만' 글자, 2 토큰화와 임베딩은 문자 기, 대, 안이 토큰 ID 18, 47, 73을 거쳐 파란 토큰 벡터가 되는 장면(아래 라벨 '문자', '토큰 ID', '토큰 벡터'), 3 어텐션은 '은행'이 돈을 0.6, 맡긴 0.1, 은행 0.3의 보라 막대로 '은행이 모든 토큰을 어텐션 가중치로 가중합'하는 장면입니다. 4 위치 인코딩은 '6번 위치의 공백'과 '10번 위치의 공백'에서 같은 파란 토큰 벡터에 서로 다른 초록 위치 벡터(위치 인코딩 행렬의 실제 값)를 더해 서로 다른 벡터가 되는 장면이고, 아래에 '같은 토큰도 위치가 다르면 다른 벡터'라고 적혀 있습니다. 5 Multi-Head와 인코더 블록은 입력, Multi-Head Attention, 더하기 + LayerNorm, FFN, 더하기 + LayerNorm이 쌓이고 잔차 연결 화살표와 '× N'이 붙은 그림, 6 감성 분석은 [CLS]에서 부정, 긍정 막대(긍정이 더 큼)로 가는 그림입니다.">


**학습 목표** (그림의 번호와 같습니다)

1. **트랜스포머가 푸는 문제와 인코더·디코더 구조를 설명합니다.**
2. **리뷰를 문자 토큰과 토큰 ID로 바꾸고, `nn.Embedding`으로 토큰마다 벡터를 만듭니다.**
3. **어텐션이 필요한 이유를 설명하고, Query·Key·Value로 가중합을 한 번 직접 계산합니다.**
4. **어텐션이 토큰의 순서를 모른다는 것을 확인하고, sin, cos 위치 인코딩을 더해 순서를 구분하게 합니다.**
5. **`nn.MultiheadAttention`과 패딩 마스크를 쓰고, 인코더 블록(잔차 연결, LayerNorm, FFN)을 `nn.TransformerEncoder`로 쌓습니다.**
6. **인코더로 영화 리뷰 감성 분석 모델을 학습하고, 학습 전·후를 비교합니다.**

## 1. 트랜스포머란

### 1-1. 트랜스포머가 푸는 문제

문장을 이해하려면 토큰마다 주변 토큰을 함께 봐야 합니다. 같은 낱말도 문맥에 따라 뜻이 바뀌기 때문입니다.

- **문맥에 따라 바뀌는 뜻:** `나는 밤이 좋다`의 밤은 어두운 시간이고, `나는 밤을 먹었다`의 밤은 먹는 열매입니다.
  - 어느 뜻인지는 주변의 `좋다`, `먹었다`를 봐야 정해집니다.
- **차례로 읽는 방식의 한계:** 토큰을 앞에서부터 하나씩 읽으면, 앞 토큰 계산이 끝나야 다음 토큰을 계산하므로 토큰들을 동시에 계산하지 못합니다.
  - 멀리 떨어진 앞 토큰의 정보는 여러 단계를 거치며 흐려집니다.
- **트랜스포머의 방법:** 한 층 안에서 모든 토큰이 다른 모든 토큰을 한꺼번에 봅니다.
  - 멀리 있는 토큰도 한 번에 보고, 문장 전체를 GPU에서 함께 계산합니다.

토큰마다 다른 토큰을 보고 필요한 정보를 모으는 계산이 어텐션(attention)입니다. 6절에서 만듭니다.

### 1-2. 전체 구조: 인코더와 디코더

**트랜스포머(transformer)**
- 어텐션 층과 FFN을 잔차 연결과 LayerNorm으로 감싼 블록을 여러 층 쌓은 모델입니다.
- 블록의 부품은 이 교안에서 하나씩 만듭니다.
- **인코더(encoder):** 입력 토큰을 모두 보고, 토큰마다 입력 전체의 정보를 반영한 벡터를 냅니다.
  - 문장 분류처럼 입력을 이해하는 일을 맡습니다.
- **디코더(decoder):** 지금까지 만든 토큰만 보고 다음 토큰의 확률을 냅니다.
  - 마스크로 뒤 토큰을 보지 못하게 막습니다.


<img src="images/19_transformer_architecture.png" width="1050" alt="제목은 '트랜스포머 전체 구조: 인코더와 디코더'입니다. 왼쪽 파란 판은 인코더, 오른쪽 주황 판은 디코더입니다. 인코더는 아래 '입력 문장의 토큰' 칸 4개('나는', '은행에', '돈을', '맡겼다')에서 올라와 '토큰 임베딩 + 위치 인코딩' 상자를 지나고, 굵은 테두리 블록 안에서 Multi-Head Attention, 더하기 + LayerNorm, FFN, 더하기 + LayerNorm 순서로 올라가며 블록 옆에 '× N'이 붙어 있습니다. 디코더는 '지금까지 만든 토큰 (번역 예)' 칸 4개('[시작]', 'I', 'put', 'money')에서 올라와 '토큰 임베딩 + 위치 인코딩' 상자를 지나고, Masked Multi-Head Attention, 더하기 + LayerNorm, Cross Attention, 더하기 + LayerNorm, FFN, 더하기 + LayerNorm 을 거쳐 Linear + softmax, 다음 토큰 확률로 올라갑니다. 층마다 입력에서 갈라진 잔차 연결 화살표가 바로 위 '더하기 + LayerNorm' 옆면으로 들어가고, 인코더 출력 선이 디코더의 Cross Attention 으로 이어집니다. 아래 띠는 '문장 이해: 분류 · BERT 계열'과 '글 생성: 번역 (GPT는 Cross Attention 없이)'입니다.">


디코더는 개념만 다루고, 이 교안은 인코더를 만듭니다.

인코더와 디코더를 어떻게 쓰는지에 따라 트랜스포머 모델은 세 갈래로 나뉩니다.

| 갈래 | 대표 모델 | 주로 하는 일 |
|---|---|---|
| Encoder-only | BERT | 문장 분류, 문장 임베딩 |
| Decoder-only | GPT, Llama, Qwen | 다음 토큰을 이어 만드는 생성 |
| Encoder-Decoder | 원 논문의 번역 모델, T5 | 번역, 요약 |

## 2. 실습 준비와 영화 리뷰 데이터

### 2-1. 실습 준비

준비 셀은 PyTorch 신경망 단원과 같습니다(데이터 경로, 모델 파일 경로, 계산 장치, 그래프 글꼴). 여기에 두 가지를 더합니다.

- 신경망 층을 담은 `torch.nn`을 불러옵니다.
- DataFrame의 긴 문자열을 자르지 않게 설정합니다.

In [ ]:
import platform
import sys
from pathlib import Path

import pandas as pd
import matplotlib.pyplot as plt
import torch
import torch.nn as nn

material_dir = Path(".")  # data가 있는 단원 폴더. 정답 노트북에서는 ".."
data_dir = material_dir / "data"

# 단원 폴더의 모델 파일(encoder_models.py)을 import 할 수 있게 경로에 더합니다
sys.path.append(str(material_dir))

# 계산 장치: RunPod처럼 GPU(cuda)가 있으면 GPU, 없으면 CPU에서 계산합니다
if torch.cuda.is_available():
    device = torch.device("cuda")
    print("계산 장치:", device, "|", torch.cuda.get_device_name(0))
else:
    device = torch.device("cpu")
    print("계산 장치:", device)

# 그래프 글꼴: 영문, 숫자, 기호는 DejaVu Sans, 한글은 운영체제의 한글 글꼴이 맡습니다
from matplotlib import font_manager

if platform.system() == "Windows":
    korean_font = "Malgun Gothic"
elif platform.system() == "Darwin":
    korean_font = "AppleGothic"
else:
    korean_font = "NanumGothic"
    # 리눅스(RunPod): 글꼴을 설치하기 전에 만든 matplotlib 글꼴 목록에는 나눔고딕이 없으므로, 설치된 파일을 직접 등록합니다
    for font_path in font_manager.findSystemFonts():
        if "NanumGothic" in font_path:
            font_manager.fontManager.addfont(font_path)
plt.rcParams["font.family"] = ["DejaVu Sans", korean_font]
plt.rcParams["axes.unicode_minus"] = False  # 음수 부호가 네모로 깨지지 않게 합니다


# DataFrame의 긴 문자열(리뷰 문장, 숫자 20개)을 자르지 않고 모두 보여 줍니다
pd.set_option("display.max_colwidth", None)

### 2-2. 영화 리뷰 데이터 읽기

- **출처:** [네이버 영화 리뷰 감성 말뭉치 NSMC](https://github.com/e9t/nsmc)(Lucy Park, CC0 1.0)
- **크기:** 여기서 고른 학습용 50,000개와 테스트용 5,000개입니다. 학습용과 같은 리뷰는 테스트용에서 뺐습니다.
- **정답:** 평점 1~4점 리뷰는 부정(0), 9~10점 리뷰는 긍정(1)으로 표시되어 있습니다.

학습용과 테스트용 파일을 읽고 앞 5줄을 봅니다.

In [ ]:
# 리뷰 문장과 감성(0 부정, 1 긍정) 두 열입니다
nsmc_train = pd.read_csv(data_dir / "본실습" / "nsmc_train.csv")
nsmc_test = pd.read_csv(data_dir / "본실습" / "nsmc_test.csv")
print("학습용 표 shape:", nsmc_train.shape, "| 테스트용 표 shape:", nsmc_test.shape)
display(nsmc_train.head())

긍정 비율과 리뷰 길이를 요약합니다.

In [ ]:
# 긍정 비율과 리뷰 길이(공백 문자를 포함한 문자 수)
review_lengths = nsmc_train["리뷰"].str.len()
nsmc_summary = pd.Series({
    "긍정 비율": nsmc_train["감성"].mean(),
    "길이 중앙값": review_lengths.median(),
    "길이 90% 지점": review_lengths.quantile(0.9),
    "가장 긴 길이": review_lengths.max(),
})
display(nsmc_summary.round(4))

학습용 50,000개를 섞어 학습 45,000개와 검증 5,000개로 나눕니다.

In [ ]:
# 학습용 50,000개를 섞어 45,000개는 학습, 5,000개는 검증에 씁니다(섞는 순서 고정)
shuffled = nsmc_train.sample(frac=1, random_state=0)
train_table = shuffled.iloc[:45000]
valid_table = shuffled.iloc[45000:]
print("학습:", len(train_table), "| 검증:", len(valid_table), "| 테스트:", len(nsmc_test))

**결과 읽기**
- **긍정 비율 0.4972:** 긍정과 부정이 거의 반반이라, 무엇이든 한쪽으로만 답하는 모델의 정확도는 약 0.5입니다.
- **길이 중앙값 문자 27개, 90% 지점 문자 77개, 가장 긴 리뷰 문자 158개:** 짧은 리뷰가 대부분이고 긴 리뷰가 조금 있습니다.
  - 모든 리뷰를 같은 시퀀스 길이로 맞출 때 이 분포로 길이를 정합니다.
- **학습 45,000개, 검증 5,000개:** 쓰임은 PyTorch 신경망 단원과 같습니다.
  - 검증 데이터는 에폭마다 성능 보기에 씁니다.
  - 테스트 데이터는 학습을 마친 모델에 한 번만 씁니다.

## 3. 토큰화와 어휘 사전

모델은 문자를 그대로 계산하지 못합니다. 리뷰를 토큰으로 자르고, 토큰마다 정수 토큰 ID를 붙여 숫자로 바꿉니다.

테스트 리뷰 하나(`기대안하고 봤는데 정말 잘 봤습니다`)로 단계마다 값과 shape을 확인하며 따라갑니다.

### 3-1. 토큰화

**토큰화(tokenization)**
- 문장을 모델이 다루는 단위인 토큰(token)의 목록으로 자르는 일입니다.
- 이 교안은 문자 하나를 토큰 하나로 자릅니다. 이를 문자 단위 토큰화(character-level tokenization)라고 합니다.
- 공백 문자 `' '`도 토큰 하나입니다.
- 실제로 쓰는 한글 문자는 수천 개 정도라, 학습 리뷰에서 본 문자로 처음 보는 리뷰도 거의 다 나타낼 수 있습니다.

예시 리뷰를 문자 단위로 자르면 다음과 같습니다.


<img src="images/34a_sentiment_tokens.png" width="1050" alt="제목은 '토큰화: 리뷰를 문자 토큰으로 자릅니다'입니다. 위 흰 상자의 리뷰 '기대안하고 봤는데 정말 잘 봤습니다'에서 'list(문장)' 화살표가 내려와 문자 토큰 카드 19장이 한 줄로 놓이고, 카드 아래에 위치 0~18이 적혀 있으며 공백 카드에는 '공백'이라고 쓰여 있습니다. 위치 6과 15의 '봤' 카드는 주황 굵은 테두리이고 아래 주황 괄호에 '같은 문자 '봤'이 두 번 나옵니다'가 붙어 있습니다. 그 아래 상자에는 '문자 19개 = 토큰 19개 (공백 4개 포함)'와 list(문장) 출력 앞부분 ['기', '대', '안', '하', '고', ' ', '봤', '는', '데', ' ', ...]가 있고, 아래 띠에는 '문자 단위 토큰화: list(문장)', '공백 문자 ' '도 토큰 하나입니다'가 적혀 있습니다.">

In [ ]:
# 테스트용 리뷰 하나로 단계마다 값을 따라갑니다
sample_review = "기대안하고 봤는데 정말 잘 봤습니다"

# 문자 단위 토큰화: 공백 문자(" ")도 토큰 하나입니다
char_tokens = list(sample_review)
print("문자 토큰", len(char_tokens), "개:", char_tokens)

**결과 읽기**
- **문자 토큰 19개:** 공백 4개(`' '`)도 토큰입니다.

### 3-2. 어휘 사전과 토큰 ID

5절의 임베딩 행렬에서 토큰마다 벡터를 꺼내려면 토큰마다 정수 ID가 있어야 합니다. 문자마다 토큰 ID를 정해 둔 어휘 사전을 만듭니다.

**어휘 사전(vocabulary)**
- 모델이 아는 토큰 목록이고, 목록의 인덱스가 토큰 ID(token id)입니다.
- 학습 데이터로만 만듭니다. 검증이나 테스트 데이터로 만들면 평가할 데이터의 정보가 학습에 새어 들어갑니다.
- 앞에 특수 토큰 세 개를 둡니다.
  - **`[PAD]` = 0:** 짧은 리뷰의 시퀀스 길이를 맞추려고 뒤에 채우는 토큰입니다.
  - **`[UNK]` = 1:** 어휘 사전에 없는 토큰(unknown)을 대신하는 토큰입니다.
  - **`[CLS]` = 2:** 리뷰 맨 앞에 붙여 분류에 쓰는 토큰입니다(10절).

예시 리뷰의 문자마다 어휘 사전에서 토큰 ID를 찾으면 다음과 같습니다.


<img src="images/34b_sentiment_ids.png" width="1050" alt="제목은 '어휘 사전에서 문자마다 토큰 ID 찾기'입니다. 위 줄의 토큰 카드 19장(공백은 '공백')에서 화살표가 아래 줄 토큰 ID 18, 47, 73, 13, 8, 3, 89, 7, 44, 3, 43, 53, 3, 134, 3, 89, 135, 34, 6으로 내려가고, 두 '봤'과 그 토큰 ID 89가 주황 테두리이며 '같은 문자 '봤'은 두 번 모두 토큰 ID 89'라고 적혀 있습니다. 왼쪽 아래 상자 '어휘 사전 vocab (어휘 크기 1,697) 일부'에는 토큰 ID와 토큰이 0 [PAD], 1 [UNK], 2 [CLS], 3 공백, ..., 18 기, ..., 89 봤(주황) 순으로 있습니다. 오른쪽 아래 상자 'token_to_id: 토큰 -> 토큰 ID'에는 token_to_id['기'] = 18, token_to_id[' '] = 3 (공백), token_to_id['봤'] = 89(주황), '어휘 사전에 없는 문자 -> 1 ([UNK])'이 있습니다. 아래 띠에는 'token_to_id[토큰] = 토큰 ID', '같은 문자는 같은 토큰 ID, 어휘 사전에 없는 문자는 [UNK](1)'이 적혀 있습니다.">

In [ ]:
from collections import Counter

# 학습 리뷰에서 문자마다 나온 횟수를 셉니다
char_counts = Counter()
for review in train_table["리뷰"]:
    char_counts.update(list(review))

# 3번 이상 나온 문자만 어휘 사전에 넣습니다(한두 번 나온 문자는 임베딩 벡터를 학습할 예가 너무 적습니다)
min_count = 3

# 특수 토큰 세 개 다음에, min_count번 이상 나온 문자를 많이 나온 순서로 붙입니다
vocab = ["[PAD]", "[UNK]", "[CLS]"]
for token, count in char_counts.most_common():
    if count >= min_count:
        vocab.append(token)

# 토큰 -> 토큰 ID를 찾는 딕셔너리
token_to_id = {}
for token_id, token in enumerate(vocab):
    token_to_id[token] = token_id
print("어휘 크기:", len(vocab))
print("앞 13개:", vocab[:13])

`encode`는 문장의 문자마다 토큰 ID를 찾는 함수입니다. 어휘 사전에 없는 문자는 `[UNK]`의 토큰 ID 1로 바꿉니다.

In [ ]:
def encode(text, token_to_id):
    """문장을 문자로 자르고, 문자마다 토큰 ID를 찾습니다. 어휘 사전에 없는 문자는 [UNK]의 토큰 ID입니다."""
    ids = []
    for token in list(text):
        if token in token_to_id:
            ids.append(token_to_id[token])
        else:
            ids.append(token_to_id["[UNK]"])
    return ids


# 예시 리뷰의 토큰과 토큰 ID를 나란히 봅니다
sample_ids = encode(sample_review, token_to_id)
sample_id_table = pd.DataFrame({"토큰": char_tokens, "토큰 ID": sample_ids}).T
display(sample_id_table)

**결과 읽기**
- **어휘 크기(vocab size) 1,697:** 학습 리뷰에 3번 이상 나온 문자 1,694개에 특수 토큰 3개를 더한 수입니다.
- **공백 `' '`의 토큰 ID 3:** 학습 리뷰에서 가장 많이 나온 토큰이라 특수 토큰 바로 다음 ID입니다.
- **같은 문자 `봤`:** 두 번 나오고 두 번 모두 토큰 ID 89입니다.
- 토큰 ID는 이름표일 뿐이라 ID 값의 크기에는 뜻이 없습니다. 뜻은 5절의 토큰 임베딩이 학습합니다.

## 4. 시퀀스 길이 맞추기: 자르기와 패딩

미니배치는 shape (리뷰 수, 토큰 수)인 텐서 하나라서, 리뷰마다 길이가 다른 토큰 시퀀스(sequence)를 같은 길이로 맞춥니다.

- **`max_len`:** 한 리뷰의 시퀀스 길이(토큰 수) 상한입니다. 이 교안은 [CLS]를 포함해 64개로 정합니다.
  - 시퀀스가 길수록 모델의 계산이 많아지므로, 대부분의 리뷰가 들어가는 길이를 고릅니다.
- **자르기(truncation):** [CLS]를 붙인 길이가 `max_len`을 넘으면 뒤를 버립니다.
- **패딩(padding):** 짧으면 뒤를 `[PAD]`(0)로 채웁니다.

예시 리뷰는 짧아서 [CLS]를 붙이고 뒤를 [PAD]로 채웁니다.


<img src="images/34c_sentiment_padding.png" width="1050" alt="제목은 '[CLS]와 [PAD]: 시퀀스 길이를 64로 맞춥니다'입니다. 위쪽에 토큰 줄, 토큰 ID 줄, 위치 줄이 있습니다. 위치 0은 주황 [CLS](토큰 ID 2), 위치 1~19는 문자 토큰(공백은 '공백', 토큰 ID 18, 47, 73, ..., 6), 위치 20~22와 63은 회색 [PAD](토큰 ID 0)이고 그 사이는 '...'입니다. 토큰 줄 위 괄호에 '[CLS] 1', '문자 19', '[PAD] 44'가 붙어 있습니다. 가운데에는 '64개 위치 전체를 길이 비율대로 그리면' 아래 막대가 [CLS] 한 칸, 파란 '문자 19', 회색 '[PAD] 44'로 나뉘고 눈금 0, 20, 64가 있습니다. 아래 띠에는 'shape (1, 64): [CLS] 1 + 문자 19 + [PAD] 44', '[CLS]는 토큰 ID 2, [PAD]는 토큰 ID 0, 리뷰마다 길이가 달라도 64개 위치로 맞춥니다'가 적혀 있습니다.">

In [ ]:
max_len = 64  # [CLS] 포함 토큰 수 상한


def make_tensor(texts, token_to_id, max_len):
    """문장마다 [CLS] + 문자 토큰 ID 시퀀스를 만들고, max_len에 맞춰 자르거나 [PAD](0)로 채운 shape (문장 수, max_len) 텐서를 돌려줍니다."""
    rows = []
    for text in texts:
        ids = [token_to_id["[CLS]"]] + encode(text, token_to_id)
        ids = ids[:max_len]  # 길면 뒤를 자릅니다
        ids = ids + [token_to_id["[PAD]"]] * (max_len - len(ids))  # 짧으면 뒤를 [PAD](0)로 채웁니다
        rows.append(ids)
    return torch.tensor(rows)


# 예시 리뷰 하나를 텐서로 바꿉니다
sample_tensor = make_tensor([sample_review], token_to_id, max_len)
print("예시 리뷰 텐서 shape:", tuple(sample_tensor.shape))
print("앞 24개 토큰 ID:", sample_tensor[0, :24].tolist())

# 학습 리뷰 가운데 잘리는 리뷰의 비율([CLS] 1개를 더한 길이가 max_len을 넘는 리뷰)
truncated_ratio = ((train_table["리뷰"].str.len() + 1) > max_len).mean()
print("잘리는 리뷰 비율:", round(truncated_ratio, 4))

**결과 읽기**
- **shape (1, 64):** 맨 앞 2가 [CLS], 이어서 문자 토큰 19개의 ID, 나머지 위치 44개가 0([PAD])입니다.
- **잘리는 리뷰 0.1323:** 학습 리뷰의 약 13%만 뒤가 잘립니다.

학습, 검증, 테스트 리뷰 전체도 같은 함수로 바꿉니다.

In [ ]:
# 리뷰 전체를 (리뷰 수, 64) 텐서로 바꾸고, 정답은 0/1 정수 텐서로 꺼냅니다
X_train = make_tensor(train_table["리뷰"].tolist(), token_to_id, max_len)
y_train = torch.tensor(train_table["감성"].values)
X_valid = make_tensor(valid_table["리뷰"].tolist(), token_to_id, max_len)
y_valid = torch.tensor(valid_table["감성"].values)
X_test = make_tensor(nsmc_test["리뷰"].tolist(), token_to_id, max_len)
y_test = torch.tensor(nsmc_test["감성"].values)
print("학습 텐서 shape:", tuple(X_train.shape), "| 검증 텐서 shape:", tuple(X_valid.shape), "| 테스트 텐서 shape:", tuple(X_test.shape))

## 5. 토큰 임베딩

토큰 ID는 이름표라서 값의 크기에 뜻이 없습니다. 모델이 계산하려면 토큰마다 뜻을 담을 수 있는 벡터가 필요합니다.

**토큰 임베딩(token embedding)**
- **`nn.Embedding(어휘 크기, d_model)`:** shape (어휘 크기, d_model)인 임베딩 행렬을 가진 층입니다. 임베딩 행렬은 학습되는 파라미터이고, 행 하나가 토큰 하나의 벡터입니다.
- **꺼내기:** 토큰 ID를 넣으면 임베딩 행렬에서 그 ID 번째 행을 꺼냅니다. 같은 토큰 ID는 늘 같은 벡터입니다.
- **학습:** 임베딩 행렬의 값은 무작위로 시작하고, 다른 층의 가중치처럼 역전파로 학습됩니다.
- **`padding_idx=0`:** [PAD](ID 0)의 행은 0으로 두고 학습하지 않습니다.
- **d_model:** 토큰 벡터의 차원입니다. 이 교안은 64차원입니다.


<img src="images/36_embedding_lookup.png" width="1050" alt="제목은 '토큰 임베딩: 토큰 ID로 임베딩 행렬의 한 행을 꺼냅니다'입니다. 위쪽 띠 '리뷰 '기대안하고 봤는데 정말 잘 봤습니다' -> make_tensor -> 토큰 ID, shape (1, 64)'에는 토큰 [CLS], 기, 대, 안, 하, 고, 공백, 봤, 는, 데, 공백, 정과 토큰 ID 2, 18, 47, 73, 13, 8, 3, 89, 7, 44, 3, 43이 있습니다. 가운데 'token_embedding.weight (1697 x 64)'는 빨강, 흰색, 파랑 칸 줄의 행들(0 [PAD], 1 [UNK], 2 [CLS], 3 공백, 4 ., ..., 17 나, 18 기, 19 한, ..., 1696)이고 0번 행은 흰색이며 아래에 '0번 행 [PAD]는 모두 0 (padding_idx=0)'이 있습니다. 왼쪽 상자 '0번 위치 [CLS]', '6번 위치 공백', '10번 위치 공백', '1번 위치 기'에서 'ID 2', 'ID 3', 'ID 3', 'ID 18' 주황 화살표가 각 행으로 가고, 꺼낸 2, 3, 18번 행이 오른쪽 '[CLS]의 토큰 벡터', '공백의 토큰 벡터 (6번, 10번 위치)', '기의 토큰 벡터'가 되며 아래에 '64차원 = d_model'이 있습니다. 아래 띠에는 'nn.Embedding(1697, 64): 토큰 ID의 행(64차원)을 임베딩 행렬에서 꺼냅니다', '같은 토큰 ID는 늘 같은 행, 행의 값은 무작위로 시작해 역전파로 학습됩니다', '색은 학습 전 임베딩 행렬의 값 (빨강 +, 파랑 -, 흰색 0)'이 적혀 있습니다.">


공백(토큰 ID 3)은 예시 리뷰의 6번, 10번, 13번, 15번 위치에 있고, 네 위치 모두 임베딩 행렬의 같은 3번 행을 꺼냅니다.

임베딩 행렬을 만들고 예시 리뷰의 토큰 ID (1, 64)를 토큰 벡터로 바꿉니다.

In [ ]:
# 임베딩 행렬 (어휘 크기, 64). [PAD](ID 0)의 행은 0으로 둡니다
d_model = 64
torch.manual_seed(0)
token_embedding = nn.Embedding(len(vocab), d_model, padding_idx=0)
print("토큰 임베딩 shape:", tuple(token_embedding.weight.shape))

# 토큰 ID마다 임베딩 행렬의 행을 꺼냅니다
with torch.no_grad():
    token_vectors = token_embedding(sample_tensor)  # (1, 64) -> (1, 64, 64)
print("토큰 ID shape:", tuple(sample_tensor.shape), "-> 토큰 벡터 shape:", tuple(token_vectors.shape))

# 0번 위치([CLS])의 벡터는 임베딩 행렬에서 그 토큰 ID의 행과 같습니다
first_id = sample_tensor[0, 0].item()
print("0번 위치 토큰 ID:", first_id, "| 임베딩 행렬의 행과 같은가:", torch.equal(token_vectors[0, 0], token_embedding.weight[first_id]))

# [PAD](ID 0)의 행은 모두 0입니다
print("[PAD] 행이 모두 0인가:", torch.all(token_embedding.weight[0] == 0).item())

**결과 읽기**
- **0번 위치 토큰 ID 2, 임베딩 행렬의 행과 같은가 True:** 임베딩은 토큰 ID로 임베딩 행렬의 행을 꺼내는 일입니다.
- **[PAD] 행이 모두 0인가 True:** `padding_idx=0` 덕분입니다.
- 지금 값은 무작위이고, 10절에서 감성 분석 모델과 함께 학습됩니다.

## 6. 어텐션

### 6-1. 어텐션이 필요한 이유

토큰 임베딩은 토큰 ID마다 벡터가 하나뿐입니다. 그래서 같은 문자는 어느 문장에 있든 같은 벡터입니다.

1-1의 두 문장에서 `밤`의 토큰 벡터를 꺼내 비교합니다.

In [ ]:
# 두 문장을 텐서로 바꾸고 토큰 벡터를 꺼냅니다
context_sentences = ["나는 밤이 좋다", "나는 밤을 먹었다"]
context_tensor = make_tensor(context_sentences, token_to_id, max_len)  # (2, 64)
with torch.no_grad():
    context_vectors = token_embedding(context_tensor)  # (2, 64) -> (2, 64, 64)

# 문장마다 '밤'의 위치. [CLS]가 0번 위치라서 문장 안 순서에 1을 더합니다
first_position = list(context_sentences[0]).index("밤") + 1
second_position = list(context_sentences[1]).index("밤") + 1
print("'밤'의 위치:", first_position, second_position)
print("'밤'의 토큰 ID:", context_tensor[0, first_position].item(), context_tensor[1, second_position].item())
print("두 토큰 벡터가 같은가:", torch.equal(context_vectors[0, first_position], context_vectors[1, second_position]))

**결과 읽기**
- **두 토큰 벡터가 같은가 True:** 두 문장의 `밤`은 토큰 ID가 811로 같아 토큰 벡터도 같습니다.
- 토큰 임베딩만으로는 어두운 시간인 밤과 먹는 밤을 가를 수 없습니다.

어텐션은 현재 토큰과 주변 토큰의 벡터를 가중합해(블렌더로 섞듯) 문맥이 담긴 새 벡터를 만듭니다.

- **넣는 것:** 현재 토큰(`밤`)을 포함한 문장의 모든 토큰 벡터
- **가중합 비율:** 어텐션 가중치. 토큰의 내용을 비교해 정하고, 뜻을 정하는 데 중요한 토큰(`좋다`, `먹었다`)일수록 큽니다.
- **나오는 것:** 가중합(weighted sum)한 벡터. 같은 `밤`이라도 문장마다 다른 벡터가 됩니다.


<img src="images/21_context_blender.png" width="1050" alt="제목은 '같은 단어, 다른 뜻: 어텐션은 주변 토큰을 가중합해 문맥을 담습니다'입니다. 위아래 두 구역에 문장 A '나는 밤이 좋다'와 문장 B '나는 밤을 먹었다'가 있고, 구역마다 왼쪽에 토큰 카드(A: 나는, 밤, 이, 좋다 / B: 나는, 밤, 을, 먹었다)와 그 옆의 파란 토큰 벡터가 세로로 놓여 있습니다. '밤' 카드는 주황 굵은 테두리이고 두 문장의 '밤' 토큰 벡터는 같은 주황 칸 줄이며, 왼쪽 바깥의 주황 점선이 두 '밤'을 잇고 두 구역 사이에 '두 '밤'의 토큰 벡터가 같습니다 (토큰 ID가 같음)'라고 적혀 있습니다. 가운데에는 뚜껑, 컵, 칼날, 받침으로 된 블렌더 그림과 '어텐션 (블렌더)' 라벨이 있고, 모든 토큰 벡터(현재 토큰 '밤' 자신 포함)에서 블렌더로 들어가는 화살표에 어텐션 가중치 0.1, 0.3, 0.1, 0.5가 적혀 있으며 '좋다'와 '먹었다'에서 오는 화살표가 가장 굵습니다. 블렌더 오른쪽에서 나온 화살표가 ''밤'의 문맥 벡터'로 가는데 문장 A는 보라 칸 줄, 문장 B는 초록 칸 줄로 서로 다르고, 그 옆 상자에 '밤 = 어두운 시간'('좋다'를 가장 많이 반영)과 '밤 = 먹는 열매'('먹었다'를 가장 많이 반영)가 적혀 있습니다. 두 구역 사이 오른쪽에는 '어텐션 뒤에는 두 '밤'의 문맥 벡터가 서로 다릅니다'가 있습니다. 아래 띠에는 '토큰 임베딩만으로는 같은 토큰이 늘 같은 벡터입니다', '어텐션은 현재 토큰을 포함한 모든 토큰 벡터를 가중합해 문맥에 맞는 벡터를 만듭니다', '어텐션 가중치는 토큰 내용을 비교해 정해집니다 (그림 값은 예시)'가 적혀 있습니다.">

그림은 보기 쉽게 낱말 단위로 그렸습니다. 실습은 문자 토큰입니다.

### 6-2. 어텐션의 원리

**어텐션(attention)**
- 토큰 하나의 새 벡터를 만들 때 입력의 모든 토큰과 비교해 어텐션 점수를 내고, 어텐션 점수가 큰 토큰에 큰 어텐션 가중치를 주어 가중합하는 계산입니다.
- **목적:** 필요한 토큰이 몇 번째 위치에 있든, 얼마나 멀리 있든 내용으로 찾아 한 번에 가져옵니다.

**Query, Key, Value**
- **Query(질의):** 지금 계산하는 토큰이 어떤 정보를 찾는지 나타낸 벡터입니다.
- **Key(키):** 각 토큰이 어떤 정보를 가졌는지 나타낸 벡터입니다. Query와 비교되는 쪽입니다.
- **Value(값):** 각 토큰이 실제로 넘겨주는 정보 벡터입니다.
- 세 벡터는 토큰 벡터에 서로 다른 가중치 행렬(학습되는 파라미터)을 곱해 만듭니다.

Query 행렬 $Q$와 Key 행렬 $K$의 어텐션 점수를 $\sqrt{d_k}$로 나눠 softmax한 어텐션 가중치로 Value 행렬 $V$를 가중합한다는 식입니다.

$\displaystyle \text{Attention}(Q,K,V)=\text{softmax}\left(\frac{QK^\top}{\sqrt{d_k}}\right)V$

- **$QK^\top$:** Query마다 모든 Key와의 어텐션 점수를 모은 행렬입니다. shape은 (Query 토큰 수, Key 토큰 수)입니다.
- **$\sqrt{d_k}$로 나누기:** $d_k$는 Key 벡터의 차원입니다. $d_k$가 클수록 내적이 커져 softmax가 한 토큰에 몰리고 기울기가 작아지는 것을 막습니다.

문장 `나는 은행에 돈을 맡겼다`에서 `은행에` 토큰의 새 벡터를 만드는 순서입니다.


<img src="images/22_attention_diagram.png" width="1050" alt="제목은 '어텐션: Query 하나를 모든 토큰의 Key와 비교해 Value를 가중합합니다'입니다. 토큰 '나는', '은행에', '돈을', '맡겼다'마다 위에서 아래로 이어지는 세로 줄이 있고, '은행에' 카드는 파란 굵은 테두리로 지금 계산하는 Query 토큰임을 보입니다. 카드 아래 회색 칸 줄은 토큰 벡터 x이고, 1 'q, k, v 만들기 / x × W_Q, W_K, W_V' 줄에서 토큰마다 자기 x에서 화살표 세 개가 파란 q, 주황 k, 초록 v 벡터로 갈라집니다('은행에'의 q만 굵은 파란 테두리). 2 '어텐션 점수 / q · k / √d_k' 줄에서는 토큰마다 1단계에서 만든 자기 k가 주황 세로 화살표로 바로 아래로 내려와 'Key: 나는', 'Key: 은행에', 'Key: 돈을', 'Key: 맡겼다'가 되고, 왼쪽의 파란 상자 'Query: 은행에의 q' 하나에서 나온 파란 가로선이 네 Key 모두에 세로로 닿으며('은행에' 자신의 Key 포함), 그 아래에 어텐션 점수 0.0, 0.7, 1.8, 0.0이 적혀 있습니다. 3 'softmax -> 어텐션 가중치' 줄에는 각 토큰 아래 보라 막대 0.1, 0.2, 0.6, 0.1이 있고 '돈을'이 가장 높으며 오른쪽에 '합 = 1'이 있습니다. 4 'Value 가중합' 줄에서는 'Value: 나는의 v'처럼 토큰마다 자기 초록 v 벡터에 '0.1 ×', '0.2 ×', '0.6 ×', '0.1 ×'가 붙어 오른쪽 연노랑 상자 '은행에의 출력'으로 화살표가 모이고, '돈을'에서 오는 화살표가 가장 굵습니다. 아래 띠에는 'Query는 지금 계산하는 토큰 하나, Key와 Value는 모든 토큰 (자기 포함)', '토큰마다 자기 Query로 같은 계산을 반복합니다 (그림은 '은행에' 하나)', '어텐션 점수와 어텐션 가중치는 예시 값입니다'가 적혀 있습니다.">


그림은 보기 쉽게 어절을 토큰 하나로 그렸습니다. 실습은 문자 토큰입니다.

1. **q, k, v 만들기:** 토큰 벡터 x마다 가중치 행렬 $W_Q$, $W_K$, $W_V$를 곱해 Query(q), Key(k), Value(v) 벡터를 만듭니다.
2. **어텐션 점수(attention score):** `은행에`의 q를 네 토큰의 k와 하나씩 내적하고 $\sqrt{d_k}$로 나눕니다. 클수록 지금 찾는 정보와 가깝습니다.
3. **어텐션 가중치(attention weight):** 어텐션 점수를 softmax 해 0~1이고 합이 1인 가중치로 바꿉니다. `돈을`의 어텐션 가중치가 가장 큽니다.
4. **가중합(weighted sum):** 어텐션 가중치를 v에 곱해 모두 더한 벡터가 `은행에`의 출력입니다. 이 벡터에 `돈을`의 정보가 가장 많이 들어갑니다.

**Self-Attention**
- query, key, value에 같은 토큰 벡터를 넣는 어텐션입니다.
- 토큰마다 같은 입력의 모든 토큰(자기 포함)을 봅니다.
- 인코더의 어텐션이 Self-Attention입니다.

### 6-3. 어텐션 직접 계산

원리를 확인하려고 어텐션을 한 번만 직접 계산합니다. 이후로는 같은 계산을 하는 `torch.nn` 모듈을 씁니다.

- **$W_Q$, $W_K$, $W_V$:** `nn.Linear(64, 64, bias=False)` 세 개입니다. 토큰 벡터에 shape (64, 64)인 가중치 행렬을 곱합니다.
- **`K.transpose(-2, -1)`과 `Q @ K_t`:** `@`는 행렬 곱입니다. K의 마지막 두 축을 바꿔 (배치, 64, Key 토큰 수)인 `K_t`로 만들고 Q와 곱하면, Query마다 모든 Key와 내적한 어텐션 점수 행렬 (배치, Query 토큰 수, Key 토큰 수)가 나옵니다.
- **$d_k$:** Key 벡터의 차원으로, 여기서는 d_model과 같은 64입니다. 어텐션 점수를 64의 제곱근 8로 나눕니다.

가중치 행렬 세 개를 만들고, 6-2의 네 단계를 함수 `self_attention`으로 묶습니다. 6-4와 7-3에서도 이 함수를 씁니다.

In [ ]:
# Query, Key, Value를 만드는 가중치 행렬 세 개(학습되는 파라미터)
torch.manual_seed(0)
W_Q = nn.Linear(d_model, d_model, bias=False)
W_K = nn.Linear(d_model, d_model, bias=False)
W_V = nn.Linear(d_model, d_model, bias=False)
d_k = d_model  # Key 벡터의 차원


def self_attention(x):
    """토큰 벡터 x (배치, 토큰 수, d_model)로 Self-Attention 출력과 어텐션 가중치를 계산합니다."""
    # 1. q, k, v 만들기: 토큰 벡터마다 세 가중치 행렬을 곱합니다
    #    Self-Attention이라 Q, K, V 모두 같은 x에서 만들어 Query 토큰 수 = Key 토큰 수 = 토큰 수입니다
    Q = W_Q(x)  # (배치, 토큰 수, d_model) @ (d_model, d_model) -> (배치, Query 토큰 수, d_model)
    K = W_K(x)  # (배치, Key 토큰 수, d_model)
    V = W_V(x)  # (배치, Key 토큰 수, d_model)

    # 2. 어텐션 점수: Query마다 모든 Key와 내적하고 d_k의 제곱근으로 나눕니다
    K_t = K.transpose(-2, -1)  # 마지막 두 축을 바꿉니다: (배치, Key 토큰 수, d_model) -> (배치, d_model, Key 토큰 수)
    scores = Q @ K_t / d_k ** 0.5  # (배치, Query 토큰 수, d_model) @ (배치, d_model, Key 토큰 수) -> (배치, Query 토큰 수, Key 토큰 수)

    # 3. 어텐션 가중치: Query마다 Key 토큰 축(마지막 축)으로 softmax. 한 행의 합이 1이고 shape은 그대로 (배치, Query 토큰 수, Key 토큰 수)
    weights = torch.softmax(scores, dim=-1)

    # 4. 가중합: 어텐션 가중치로 Value를 가중합합니다
    output = weights @ V  # (배치, Query 토큰 수, Key 토큰 수) @ (배치, Key 토큰 수, d_model) -> (배치, Query 토큰 수, d_model)
    return output, weights

짧은 리뷰 `나쁘지않아`의 [CLS]와 문자 토큰 5개, 모두 토큰 6개로 계산합니다.

In [ ]:
# 짧은 리뷰의 앞 위치 6개([CLS]와 문자 토큰 5개)를 토큰 벡터로 바꿉니다
short_review = "나쁘지않아"
short_tokens = ["[CLS]"] + list(short_review)
short_ids = make_tensor([short_review], token_to_id, max_len)[:, :6]  # (1, 6)
with torch.no_grad():
    x = token_embedding(short_ids)  # (1, 6) -> (1, 6, 64)
    attention_output, attention_weights = self_attention(x)  # (1, 6, 64) -> 출력 (1, 6, 64), 어텐션 가중치 (1, 6, 6)
print("입력 shape:", tuple(x.shape), "| 어텐션 가중치 shape:", tuple(attention_weights.shape), "| 출력 shape:", tuple(attention_output.shape))

# 어텐션 가중치 DataFrame: 행은 Query 토큰, 열은 Key 토큰
weight_table = pd.DataFrame(attention_weights[0].numpy(), index=short_tokens, columns=short_tokens)
display(weight_table.round(3))
print("행마다 합:", weight_table.sum(axis=1).round(4).tolist())

**결과 읽기**
- **입력 (1, 6, 64) -> 어텐션 가중치 (1, 6, 6) -> 출력 (1, 6, 64):** 토큰마다 토큰 6개의 Value를 가중합한 새 벡터가 나오고, shape은 입력과 같습니다.
- **행마다 합 1.0:** 한 행은 Query 토큰 하나가 토큰 6개에 준 어텐션 가중치입니다.
- $W_Q$, $W_K$, $W_V$가 아직 무작위라 어텐션 가중치에 뜻은 없습니다. 학습하면 필요한 토큰에 큰 어텐션 가중치를 주도록 세 행렬이 바뀝니다.

### 6-4. 어텐션은 순서를 모른다

어텐션은 Query와 Key의 내용으로만 어텐션 점수를 내고, 토큰이 몇 번째 위치인지는 쓰지 않습니다.

같은 토큰 6개를 순서만 거꾸로 바꿔 넣고, 나온 출력을 다시 원래 순서로 돌려 비교합니다.

In [ ]:
# 토큰 순서를 거꾸로 바꿉니다: 위치 5, 4, 3, 2, 1, 0
reverse_order = [5, 4, 3, 2, 1, 0]
with torch.no_grad():
    reversed_output, _ = self_attention(x[:, reverse_order])

# 거꾸로 한 번 더 놓으면 원래 순서입니다
print("순서를 바꿔 넣은 출력을 되돌리면 원래 출력과 같은가:",
      torch.allclose(reversed_output[:, reverse_order], attention_output, atol=1e-6))

**결과 읽기**
- **True:** 순서를 바꿔 넣어도 토큰마다 같은 출력이 나오고, 출력의 순서만 바뀝니다.
- 어텐션은 토큰을 순서 없는 묶음으로 봅니다. `나쁘지않아`와 `아않지쁘나`가 토큰마다 같은 결과를 냅니다.

## 7. 위치 인코딩

### 7-1. 위치 인코딩이 필요한 이유

문장은 같은 토큰이라도 순서가 바뀌면 뜻이 바뀝니다.

- **같은 낱말, 다른 평가:** `처음엔 별로 나중엔 최고`와 `처음엔 최고 나중엔 별로`는 낱말이 같습니다. 하지만 영화가 어느 쪽으로 끝났는지가 반대입니다.
- **6-4의 결과:** 어텐션만으로는 이 두 문장을 토큰 묶음이 같은 문장으로 봅니다.

그래서 토큰마다 몇 번째 위치인지를 벡터로 알려 줍니다.

### 7-2. sin, cos 위치 인코딩

**위치 인코딩(positional encoding)**
- 위치 인덱스(0, 1, 2, ...)마다 sin, cos로 계산한 벡터를 하나씩 정해 두고, 토큰 벡터에 그 위치의 벡터를 더하는 방법입니다.
- 원 논문의 방법입니다. 식으로 정해지는 고정 값이라 학습하지 않습니다.
- **목적:** 같은 토큰이라도 위치(position)에 따라 다른 벡터가 되어, 어텐션이 내용과 위치를 함께 비교합니다.

위치 $pos$의 위치 벡터에서 짝수 차원($2i$)은 sin, 홀수 차원($2i+1$)은 cos입니다.

$\displaystyle PE_{(pos,\,2i)} = \sin\left(\frac{pos}{10000^{2i/d}}\right), \quad PE_{(pos,\,2i+1)} = \cos\left(\frac{pos}{10000^{2i/d}}\right)$

- $pos$는 위치 인덱스, $d$는 d_model(64)입니다.
- 나누는 값 $10000^{2i/d}$는 뒤쪽 차원일수록 커집니다. 그래서 앞쪽 차원은 위치마다 빠르게, 뒤쪽 차원은 천천히 바뀝니다.
- 빠른 차원과 느린 차원의 조합이 위치마다 달라, 위치 벡터가 위치를 구분합니다.


<img src="images/25_positional_encoding.png" width="1050" alt="제목은 '위치 인코딩: 위치마다 sin, cos 값으로 만든 벡터를 더합니다'입니다. 왼쪽 구역 '위치 인코딩 행렬의 차원 하나를 위치 0~63 순서로 본 값'에는 곡선 3개가 있습니다. '0번 차원 (sin)'은 빠르게, '16번 차원 (sin)'은 중간 속도로, '32번 차원 (sin)'은 천천히 바뀌고, 주황 점선(위치 6)과 보라 점선(위치 10)이 곡선 위 점을 표시하며 가로축은 '위치 인덱스'입니다. 오른쪽 구역 '위치 인코딩 행렬의 행 = 그 위치의 위치 벡터'에는 '위치 6의 위치 벡터 (64차원)'와 '위치 10의 위치 벡터 (64차원)'가 빨강, 흰색, 파랑 칸 줄로 그려져 있고 0번, 16번, 32번 차원에 각각 -0.28, 0.56, 0.06과 -0.54, 0.84, 0.10이 적혀 있습니다. 그 아래에 '앞쪽 차원은 위치 6과 10의 값이 크게 다르고, 뒤쪽 차원은 비슷합니다', '색: 빨강 +1, 흰색 0, 파랑 -1'이 있습니다. 아래 띠에는 '짝수 차원은 sin, 홀수 차원은 cos', '앞쪽 차원은 위치마다 빠르게, 뒤쪽 차원은 천천히 바뀝니다', '학습되지 않는 고정 값이고, 토큰 벡터에 더합니다'가 적혀 있습니다.">


위치 64개마다 64차원 위치 벡터를 담은 위치 인코딩 행렬 (64, 64)를 계산하고, 위치 0~2에서 앞 6개 차원의 값을 봅니다.

In [ ]:
# 위치 인덱스 0~63을 세로 한 줄로 세웁니다
position_ids = torch.arange(max_len).unsqueeze(1)  # (64, 1)

# 짝수 차원 인덱스 2i = 0, 2, ..., 62마다 나누는 값 10000^(2i/d)
even_index = torch.arange(0, d_model, 2)  # (32,)
divisor = 10000 ** (even_index / d_model)  # (32,)

# 짝수 차원(0::2)은 sin, 홀수 차원(1::2)은 cos
# 브로드캐스팅: (64, 1) / (32,) -> (64, 32). 위치마다 짝수 차원 32개의 값을 한 번에 계산합니다
position_table = torch.zeros(max_len, d_model)  # (64, 64)
position_table[:, 0::2] = torch.sin(position_ids / divisor)  # (64, 32) -> 짝수 열 32개
position_table[:, 1::2] = torch.cos(position_ids / divisor)  # (64, 32) -> 홀수 열 32개
print("위치 인코딩 행렬 shape:", tuple(position_table.shape))
display(position_table[:3, :6].round(decimals=3))

**결과 읽기**
- **위치 인코딩 행렬 (64, 64):** 위치 64개마다 64차원 위치 벡터가 하나씩 있습니다.
- **0번 위치 [0, 1, 0, 1, 0, 1]:** sin 0 = 0, cos 0 = 1입니다.
- **1번, 2번 위치:** 앞쪽 차원의 값이 위치마다 달라집니다.
- `encoder_models.py`의 `positional_encoding` 함수가 이 셀과 같은 계산이고, 10절의 모델이 이 행렬을 씁니다.

**가까운 위치끼리 내적이 큽니다**
- 두 위치 벡터의 내적은 차원 쌍마다 $\sin a \sin b + \cos a \cos b = \cos(a - b)$를 더한 값입니다.
- 그래서 내적은 두 위치의 거리로만 정해지고, 거리 0일 때 가장 큽니다(차원 쌍 32개 x 1 = 32).
- 10번 위치의 위치 벡터를 거리가 다른 위치들과 내적해 봅니다.

In [ ]:
# 10번 위치의 위치 벡터와, 10번에서 distance만큼 떨어진 위치 벡터의 내적
dot_rows = []
for distance in [0, 1, 2, 5, 10, 20, 25, 40, 53]:
    dot = position_table[10] @ position_table[10 + distance]  # (64,) @ (64,) -> 스칼라
    dot_rows.append({"거리": distance, "내적": round(dot.item(), 2)})
display(pd.DataFrame(dot_rows))

# 거리가 같으면 내적도 같습니다: 20번과 25번, 40번과 45번(둘 다 거리 5)
print("20번, 25번 내적:", round((position_table[20] @ position_table[25]).item(), 2),
      "| 40번, 45번 내적:", round((position_table[40] @ position_table[45]).item(), 2))

**결과 읽기**
- **거리 0이면 32.0, 거리가 멀수록 작아집니다(거리 1 30.92, 5 23.5, 20 18.88, 53 13.37):** 가까운 위치끼리 위치 벡터가 더 비슷합니다.
  - 차례로만 줄지는 않습니다. 거리 20은 18.88, 25는 20.13으로 다시 커집니다. 크게 보면 가까울수록 큽니다.
- **20번, 25번과 40번, 45번 모두 23.5:** 내적은 위치가 아니라 거리로만 정해집니다. 원 논문은 이 성질 덕분에 모델이 두 토큰이 몇 위치 떨어져 있는지(상대 위치)를 배우기 쉽다고 설명합니다.

### 7-3. 토큰 벡터에 더하기

6-4와 같은 실험을 위치 벡터를 더해 다시 합니다.

- 원래 순서와 거꾸로 바꾼 순서 모두, 위치 0~5의 위치 벡터를 차례로 더합니다.
- 같은 토큰이라도 놓인 위치가 바뀌면 더해지는 위치 벡터가 달라집니다.

In [ ]:
# 위치 0~5의 위치 벡터 (6, 64)를 원래 순서와 거꾸로 바꾼 순서에 똑같이 더합니다
positions = position_table[:6]  # (64, 64) -> (6, 64)
with torch.no_grad():
    position_output, _ = self_attention(x + positions)  # (1, 6, 64) + (6, 64) -> (1, 6, 64)
    reversed_position_output, _ = self_attention(x[:, reverse_order] + positions)

print("순서를 바꿔 넣은 출력을 되돌리면 원래 출력과 같은가:",
      torch.allclose(reversed_position_output[:, reverse_order], position_output, atol=1e-6))

**결과 읽기**
- **False:** 위치 벡터를 더하자 순서를 바꾼 입력의 출력이 달라졌습니다. 이제 어텐션이 순서를 구분합니다.

## 8. Multi-Head Attention과 패딩 마스크

### 8-1. nn.MultiheadAttention

**Multi-Head Attention**
- d_model을 헤드 수만큼 나눠(64차원을 헤드 4개가 16차원씩), 헤드마다 Query·Key·Value와 어텐션 가중치를 따로 계산합니다.
- 헤드들의 결과(16차원씩)를 이어 붙여 다시 64차원으로 만들고, 출력 행렬을 곱합니다.
- **목적:** 헤드마다 다른 위치를 볼 수 있어, 한 층에서 여러 관계를 함께 계산합니다.


<img src="images/24_multi_head.png" width="1050" alt="제목은 'Multi-Head Attention: 64차원을 헤드 4개가 16차원씩 나눠 봅니다'입니다. 왼쪽 파란 상자 '입력 벡터 (1, 64, 64)'에서 화살표 4개가 헤드 0~3 상자로 갈라지고, 헤드마다 'q, k, v (1, 64, 16)' 상자와 진한 칸 위치가 서로 다른 보라 격자 '어텐션 가중치 (1, 64, 64)'가 있습니다. 네 헤드의 화살표가 회색 상자 '이어 붙이기 (1, 64, 64)'로 모이고, 이어 주황 상자 '출력 행렬 out_proj (64 x 64) -> 출력 (1, 64, 64)'로 갑니다. 아래 띠에는 '헤드마다 q, k, v와 어텐션 가중치를 따로 계산해 서로 다른 위치를 봅니다', '헤드 4개의 결과(16차원씩)를 이어 붙여 다시 64차원, 출력 shape은 입력과 같습니다'가 적혀 있습니다.">


**`nn.MultiheadAttention`**
- **만들기 `nn.MultiheadAttention(embed_dim, num_heads, batch_first=True)`:** 어텐션 층입니다.
  - `embed_dim`은 토큰 벡터의 차원(d_model), `num_heads`는 헤드 수입니다. embed_dim이 헤드 수로 나누어떨어져야 합니다.
  - 층 안에 6-3의 $W_Q$, $W_K$, $W_V$에 해당하는 가중치 행렬과 출력에 한 번 더 곱하는 출력 행렬(출력 투영, `out_proj`)이 들어 있습니다.
- **`batch_first`:** 기본값이 `False`라 입력을 (토큰 수, 배치, d_model)로 읽습니다.
  - (배치, 토큰 수, d_model)로 넘길 때는 반드시 `True`로 줍니다.
  - 빠뜨려도 에러 없이 축을 잘못 읽습니다.
- **부르기 `attention(query, key, value)`:** (출력, 어텐션 가중치) 두 개를 돌려줍니다.
  - 출력은 (배치, Query 토큰 수, embed_dim)입니다.
  - 어텐션 가중치는 헤드 4개를 평균한 (배치, Query 토큰 수, Key 토큰 수)입니다.

예시 리뷰의 토큰 벡터에 위치 벡터를 더한 입력 벡터 (1, 64, 64)를 넣습니다.

In [ ]:
# 입력 벡터 = 토큰 벡터 + 위치 0~63의 위치 벡터. 위치 인코딩 행렬이 배치 축으로 브로드캐스팅됩니다
input_vectors = token_vectors + position_table  # (1, 64, 64) + (64, 64) -> (1, 64, 64)

# 어텐션 층: 64차원, 헤드 4개
torch.manual_seed(0)
attention = nn.MultiheadAttention(embed_dim=d_model, num_heads=4, batch_first=True)

# Self-Attention: query, key, value 모두 입력 벡터
with torch.no_grad():
    # 출력 (1, Query 토큰 수 64, d_model 64), 어텐션 가중치 (1, Query 토큰 수 64, Key 토큰 수 64)
    attn_out, attn_weights = attention(input_vectors, input_vectors, input_vectors)
print("입력 shape:", tuple(input_vectors.shape), "| 출력 shape:", tuple(attn_out.shape), "| 어텐션 가중치 shape:", tuple(attn_weights.shape))

# 0번 위치([CLS]) Query가 [PAD] 위치 44개(20~63번)에 준 어텐션 가중치의 합
print("[PAD] 위치에 준 어텐션 가중치 합:", round(attn_weights[0, 0, 20:].sum().item(), 3))

**결과 읽기**
- **출력 (1, 64, 64):** 출력 shape은 입력과 같습니다.
- **어텐션 가중치 (배치, Query 토큰 수, Key 토큰 수) = (1, 64, 64):** Query 토큰 64개마다 Key 토큰 64개에 준 어텐션 가중치입니다.
- **[PAD] 위치에 준 어텐션 가중치 합 0.642:** 진짜 토큰은 20개([CLS] 포함)뿐인데, 어텐션 가중치의 절반 넘게 시퀀스 길이를 맞추려고 붙인 [PAD] 위치로 갑니다.

### 8-2. 패딩 마스크

[PAD]는 리뷰의 내용이 아닌데도 어텐션의 가중합에 들어갑니다. [PAD]를 몇 개 붙였는지에 따라 결과가 달라지므로, [PAD] 위치를 보지 못하게 막아야 합니다.

**패딩 마스크(padding mask)**
- 토큰 ID가 0([PAD])인 위치가 True인 shape (배치, 토큰 수)의 참/거짓 텐서입니다.
- `nn.MultiheadAttention`에는 `key_padding_mask`로 넘깁니다. **True인 위치를 Key로 보지 않습니다**.

마스크를 만들어 넘기고, [CLS] Query가 [PAD] 위치에 준 어텐션 가중치의 합을 다시 봅니다.

In [ ]:
# 패딩 마스크: [PAD](0) 위치가 True
sample_mask = sample_tensor == 0  # (1, 64)
print("마스크의 True 수:", sample_mask.sum().item())

# 마스크와 함께 다시 계산합니다
with torch.no_grad():
    masked_out, masked_weights = attention(input_vectors, input_vectors, input_vectors, key_padding_mask=sample_mask)
print("[PAD] 위치에 준 어텐션 가중치 합:", round(masked_weights[0, 0, 20:].sum().item(), 3))

**결과 읽기**
- **True 44개:** [PAD] 위치 44개에 마스크가 켜집니다.
- **[PAD] 위치에 준 어텐션 가중치 합 0.0:** 어텐션 가중치가 진짜 토큰 20개([CLS] 포함)에만 나뉩니다. [PAD]를 몇 개 붙였든 결과가 같아집니다.
- 패딩한 텐서를 어텐션에 넣을 때는 마스크를 꼭 함께 넘깁니다.

## 9. 인코더 블록

8절까지 만든 것은 어텐션 층 하나입니다. 트랜스포머는 어텐션 뒤에 FFN을 두고, 둘을 잔차 연결과 LayerNorm으로 감싼 블록을 여러 층 쌓습니다.

**인코더 블록(encoder block)**
- [Multi-Head Self-Attention → 더하기 + LayerNorm → FFN → 더하기 + LayerNorm] 순서의 층 묶음입니다.
- 입력과 출력의 shape이 모두 (배치, 토큰 수, d_model)이라 같은 블록을 여러 층 쌓습니다.
- **FFN(feed-forward network):** 토큰마다 따로 같은 2층 신경망(이 교안은 64 → 256 → 64, ReLU)을 지나게 하는 부분입니다.
  - **목적:** 어텐션이 다른 토큰에서 모아 온 정보를 토큰 안에서 다시 계산합니다.
  - 가운데 크기는 `dim_feedforward` 인자로 정합니다(원 논문은 d_model의 4배, 512 → 2048).
- **잔차 연결(residual connection):** 어텐션과 FFN의 출력을 각각 그 입력에 더합니다($x + \text{Attention}(x)$, $x + \text{FFN}(x)$).
  - PyTorch 신경망 단원의 ResNet에서 본 잔차 연결과 같은 더하기입니다.
  - 더하려면 shape이 같아야 하므로 블록 안의 모든 층은 입력과 같은 shape을 지킵니다.


<img src="images/31_transformer_blocks.png" width="1050" alt="제목은 '인코더 블록: 어텐션과 FFN을 잔차 연결과 LayerNorm으로 감쌉니다'입니다. 아래에서 위로 파란 '입력 (배치, 토큰 수, d_model)', 보라 'Multi-Head Attention', 노랑 '더하기 + LayerNorm', 회색 'FFN', 노랑 '더하기 + LayerNorm', 주황 '출력 (배치, 토큰 수, d_model)'이 화살표로 이어집니다. 가운데 네 상자를 감싼 테두리 오른쪽에 '× N'이 있고, 왼쪽의 '잔차 연결' 선 2개가 어텐션의 입력에서 첫 번째 더하기로, FFN의 입력에서 두 번째 더하기로 들어갑니다. 아래 띠에는 '블록의 입력과 출력 shape이 같아 같은 블록을 여러 층 쌓습니다'가 적혀 있습니다.">


새 부품인 LayerNorm을 먼저 보고, `nn.TransformerEncoderLayer`로 블록을 합칩니다.

### 9-1. LayerNorm

**LayerNorm(layer normalization)**
- 토큰 벡터 하나의 d_model개 값으로 평균과 분산을 계산해 평균 0, 분산 1로 표준화하는 층입니다([Ba et al., 2016](https://arxiv.org/abs/1607.06450)).
- 표준화한 뒤 학습되는 파라미터 $\gamma$(크기)를 곱하고 $\beta$(이동)를 더합니다.
- **목적:** 잔차 연결로 더해 가며 커지거나 작아지는 토큰 벡터의 크기를 층마다 일정하게 맞춰 학습을 안정시킵니다.
- 토큰 벡터의 $i$번째 값을 $x_i$, 그 벡터 $d$개 값의 평균과 분산을 $\mu$, $\sigma^2$이라 하면 다음과 같습니다.

$\displaystyle \mu = \frac{1}{d}\sum_{i=1}^{d} x_i, \qquad \sigma^2 = \frac{1}{d}\sum_{i=1}^{d} (x_i - \mu)^2$

$\displaystyle y_i = \gamma_i \frac{x_i - \mu}{\sqrt{\sigma^2 + \epsilon}} + \beta_i$

- $\epsilon$은 0으로 나누지 않게 더하는 아주 작은 수입니다.

특징(차원)이 4개인 토큰 벡터 [2, 4, 6, 8] 하나로 계산을 따라갑니다($\gamma$ = 1, $\beta$ = 0인 처음 상태).

1. **평균:** (2 + 4 + 6 + 8) / 4 = 5입니다.
2. **분산:** 평균과의 차 [-3, -1, 1, 3]을 제곱한 [9, 1, 1, 9]의 평균, 20 / 4 = 5입니다. 표준편차는 5의 제곱근 2.2361입니다.
3. **표준화:** 차를 2.2361로 나누면 [-1.3416, -0.4472, 0.4472, 1.3416]입니다. 평균 0, 분산 1입니다.
4. **크기가 다른 토큰:** [20, 40, 60, 80]도 같은 결과가 나옵니다. 값의 크기와 상관없이 토큰 안의 상대적인 크기 관계만 남습니다.

In [ ]:
# 토큰 2개, 특징 4개. 두 번째 토큰은 첫 번째의 10배입니다
token_values = torch.tensor([[2.0, 4.0, 6.0, 8.0],
                              [20.0, 40.0, 60.0, 80.0]])

# nn.LayerNorm(특징 수): 토큰마다 특징 4개로 평균과 분산을 계산해 표준화합니다(처음 gamma = 1, beta = 0)
layer_norm = nn.LayerNorm(4)
with torch.no_grad():
    normalized = layer_norm(token_values)
display(normalized.round(decimals=4))

**결과 읽기**
- **두 토큰 모두 [-1.3416, -0.4472, 0.4472, 1.3416]:** 손으로 계산한 값과 같습니다.
- 10배 큰 토큰도 같은 결과입니다. 토큰마다 자기 값으로만 표준화하기 때문입니다.

### 9-2. LayerNorm과 BatchNorm

PyTorch 신경망 단원의 배치 정규화(BatchNorm)도 평균 0, 분산 1로 표준화한 뒤 $\gamma$, $\beta$를 쓰는 층입니다.

다른 점은 평균과 분산을 계산하는 범위 하나입니다.

- **BatchNorm:** 특징 하나를 골라, 배치의 모든 문장과 토큰에 걸쳐 계산합니다(그림의 세로 줄).
- **LayerNorm:** 토큰 하나를 골라, 그 토큰의 모든 특징으로 계산합니다(그림의 가로 줄).


<img src="images/32_layernorm_vs_batchnorm.png" width="1050" alt="제목은 'BatchNorm과 LayerNorm: 평균과 분산을 내는 범위'입니다. 왼쪽 'BatchNorm' 판과 오른쪽 'LayerNorm' 판에 각각 '문장 1', '문장 2', '문장 3' 카드가 나란히 있고, 카드마다 세로 '토큰 5개', 가로 '특징 4개'인 격자입니다. BatchNorm 판은 세 카드 모두의 왼쪽에서 두 번째 열을 보라로 칠했고 아래에 '특징 하나를 모든 문장과 토큰에 걸쳐 계산'이 적혀 있습니다. LayerNorm 판은 문장 1 카드의 세 번째 행만 초록으로 칠했고 아래에 '토큰 하나의 특징 전체로 계산'이 적혀 있습니다. 맨 아래 띠에 'BatchNorm: 같은 배치의 다른 문장과 패딩이 통계에 섞입니다', 'LayerNorm: 다른 문장과 배치 크기에 영향받지 않습니다'가 적혀 있습니다.">


| 비교 | BatchNorm | LayerNorm |
|---|---|---|
| 평균과 분산을 계산하는 범위 | 특징 하나, 배치의 모든 문장과 토큰 | 토큰 하나, 그 토큰의 모든 특징 |
| 주로 쓰는 곳 | CNN(이미지) | 트랜스포머(문장) |

9-1의 두 토큰을 `nn.BatchNorm1d(4)`에도 넣어 LayerNorm 결과와 비교합니다.

In [ ]:
# BatchNorm1d(특징 수): 특징마다 배치의 모든 토큰(여기서는 2개)으로 평균과 분산을 계산합니다(학습 모드)
batch_norm = nn.BatchNorm1d(4)
with torch.no_grad():
    batch_normalized = batch_norm(token_values)

# 같은 입력의 두 결과를 나란히 봅니다
norm_table = pd.DataFrame({
    "LayerNorm 토큰 1": normalized[0].tolist(),
    "LayerNorm 토큰 2": normalized[1].tolist(),
    "BatchNorm 토큰 1": batch_normalized[0].tolist(),
    "BatchNorm 토큰 2": batch_normalized[1].tolist(),
}, index=["특징 1", "특징 2", "특징 3", "특징 4"])
display(norm_table.round(4))

**결과 읽기**
- **LayerNorm:** 두 토큰 모두 [-1.3416, -0.4472, 0.4472, 1.3416]입니다. 토큰 안의 값의 크기 관계(2 < 4 < 6 < 8)가 그대로 남습니다.
- **BatchNorm:** 토큰 1은 모두 -1, 토큰 2는 모두 1입니다. 특징마다 두 토큰을 서로 비교한 값이라, 토큰 안의 크기 관계가 사라집니다.
  - 같은 토큰이라도 배치에 어떤 토큰이 함께 있는지에 따라 결과가 바뀝니다.

**트랜스포머가 LayerNorm을 쓰는 이유**
- 문장은 길이가 제각각이라 [PAD]가 많습니다. BatchNorm은 다른 문장과 [PAD] 토큰의 값이 평균과 분산 계산에 들어가고, 그 값이 배치마다 바뀝니다.
- LayerNorm은 토큰 하나 안에서 계산하므로 다른 문장, [PAD], 배치 크기와 상관없이 같은 결과가 나옵니다.

### 9-3. nn.TransformerEncoderLayer와 nn.TransformerEncoder

부품을 직접 이어 붙이지 않아도 됩니다.

- **`nn.TransformerEncoderLayer`:** 인코더 블록 하나(Multi-Head Self-Attention, FFN, 잔차 연결 두 번, LayerNorm 두 번, Dropout)를 한 층으로 묶어 줍니다.
- **`nn.TransformerEncoder`:** 그 층을 정한 수만큼 복사해 쌓습니다.

**`nn.TransformerEncoderLayer`의 인자**
- **`d_model`:** 토큰 벡터의 차원입니다. 입력과 출력 모두 (배치, 토큰 수, d_model)입니다.
- **`nhead`:** 헤드 수입니다(8-1의 `num_heads`).
- **`dim_feedforward`:** FFN 가운데 층의 크기입니다(9절 도입의 FFN).
- **`dropout`:** 블록 안 Dropout 비율입니다(PyTorch 신경망 단원의 Dropout, 기본 0.1).
- **`batch_first=True`:** 8-1과 같습니다.

In [ ]:
# 인코더 층 하나: d_model 64, 헤드 4개, FFN 가운데 256
encoder_layer = nn.TransformerEncoderLayer(d_model=64, nhead=4, dim_feedforward=256, dropout=0.1, batch_first=True)
print(encoder_layer)

- **`self_attn`:** `nn.MultiheadAttention`입니다(`out_proj`가 헤드 결과를 합치는 층).
- **`linear1`, `linear2`:** FFN의 두 층(64 → 256 → 64)이고, 활성화 함수는 기본 ReLU입니다.
- **`norm1`, `norm2`:** 어텐션 뒤와 FFN 뒤의 LayerNorm입니다.
- **`dropout`, `dropout1`, `dropout2`:** 블록 안의 Dropout(비율 0.1)입니다.

같은 층을 2개 쌓아, 층이 늘면 파라미터가 어떻게 늘고 입력과 출력 shape이 어떤지 확인합니다.

- **`num_layers`:** 쌓을 층 수입니다. 넘겨준 층을 복사하므로 층마다 파라미터는 따로 학습됩니다.
- **파라미터 수:** PyTorch 신경망 단원에서 만든 `count_parameters`로 셉니다.

In [ ]:
# PyTorch 신경망 단원에서 만든 함수를 그대로 씁니다
def count_parameters(model):
    """모델의 파라미터(가중치, 절편, 배치 정규화의 gamma와 beta) 개수를 모두 더해 돌려줍니다."""
    total = 0
    for param in model.parameters():
        total += param.numel()
    return total


# 같은 층을 2개 쌓습니다
# enable_nested_tensor=False: 경고 없이 쓰려고 끕니다(encoder_models.py와 같게)
encoder = nn.TransformerEncoder(encoder_layer, num_layers=2, enable_nested_tensor=False)
print("층 하나:", count_parameters(encoder_layer), "| 인코더(2층):", count_parameters(encoder))

# 문장 2개, 토큰 6개, d_model 64인 입력을 넣습니다
torch.manual_seed(0)
encoder_input = torch.randn(2, 6, 64)  # (2, 6, 64)
encoder.eval()  # Dropout을 끄고 같은 입력에 같은 결과를 냅니다
with torch.no_grad():
    encoded = encoder(encoder_input)  # (2, 6, 64) -> (2, 6, 64)
print("입력 shape:", tuple(encoder_input.shape), "| 출력 shape:", tuple(encoded.shape))

**결과 읽기**
- **층 하나 49,984개, 인코더 99,968개:** 2층이면 정확히 2배입니다. 층 하나에 든 것은 다음과 같습니다.
  - 어텐션: 64 × 64 행렬 4개와 절편
  - FFN: 64 × 256 × 2와 절편
  - LayerNorm 두 개: $\gamma$, $\beta$ 64개씩
- **입력과 출력 모두 (2, 6, 64):** 블록의 입력과 출력 shape이 같아서 층을 몇 개든 쌓을 수 있습니다.

## 10. 인코더로 영화 리뷰 감성 분석

3~9절의 부품을 하나의 모델로 묶어, 영화 리뷰 한 문장이 긍정인지 부정인지 학습합니다.

### 10-1. [CLS] 벡터와 TransformerClassifier

인코더는 토큰마다 벡터 하나를 내므로, 리뷰 하나를 분류하려면 그 가운데 벡터 하나를 골라야 합니다.

**[CLS] 토큰(classification token)**
- 문장 맨 앞에 붙이는 특수 토큰이고, 인코더를 지난 이 위치의 벡터 하나로 문장 전체를 분류합니다(BERT가 쓰는 방식).
- **이 위치를 쓰는 이유**
  - [CLS]는 어떤 문자에도 속하지 않아, 학습하며 이 위치의 Query가 분류에 필요한 토큰에 큰 어텐션 가중치를 주도록 정해집니다.
  - 리뷰 길이와 상관없이 늘 0번 위치라, 길이가 다른 리뷰도 같은 위치에서 벡터 하나를 꺼냅니다.

토큰 ID가 로짓이 되는 계산 순서와 shape은 다음과 같습니다.


<img src="images/38_classifier_flow.png" width="1050" alt="제목은 'TransformerClassifier: 토큰 ID가 부정 · 긍정 로짓이 되기까지'입니다. 왼쪽 '코드', 가운데 '값과 shape', 오른쪽 '하는 일' 세 열로 위에서 아래로 진행합니다. '토큰 ID tokens (배치, 64)'(make_tensor(...), [CLS] + 문자 토큰 ID + [PAD])에서 '토큰 벡터 (배치, 64, 64)'(self.token_embedding, 토큰 ID -> 임베딩 행렬의 행)로 가고, 옆의 초록 '위치 벡터 (64, 64)'(self.position_table[:64], 위치 인코딩 행렬의 0~63행, 위치 0~63)와 + 원에서 더해져 'h = 토큰 벡터 + 위치 벡터 (배치, 64, 64)'(같은 문자도 위치마다 다른 벡터)가 됩니다. 'src_key_padding_mask tokens == 0 (배치, 64)' 상자가 보라 '인코더 (인코더 층 2개) (배치, 64, 64)'(self.encoder(h, ...), 어텐션으로 토큰 벡터를 가중합합니다)로 들어가고, 이어 '[CLS] 벡터 = 0번 위치 출력 (배치, 64)'(h[:, 0], 리뷰 하나를 벡터 하나로), 주황 '로짓 [부정 로짓, 긍정 로짓] (배치, 2)'(self.classifier, nn.Linear: 64차원 -> 클래스 2개)로 갑니다. 아래 띠에는 'softmax(로짓) = [부정 확률, 긍정 확률], 두 확률의 합은 1입니다', '학습 때 배치는 128(미니배치), 리뷰 한 문장이면 배치는 1입니다', 'd_model = 64, max_len = 64, 어휘 크기 = 1697'이 적혀 있습니다.">


모델 코드(`TransformerClassifier`)는 단원 폴더의 `encoder_models.py`에 있으므로, 불러와 코드를 펼쳐 봅니다.

In [ ]:
import inspect

from encoder_models import TransformerClassifier

# 모델 코드를 그대로 출력합니다
print(inspect.getsource(TransformerClassifier))

- **`__init__`(층):** 토큰 임베딩 행렬(5절), 위치 인코딩 행렬(7절), 인코더(9절, `num_layers` 기본 2), [CLS] 벡터 하나를 클래스 수만큼의 로짓으로 바꾸는 `self.classifier`를 만듭니다.
  - 위치 인코딩 행렬은 학습하지 않는 값이라 `register_buffer`로 등록한 버퍼(학습되지 않는 텐서)로 둡니다. 파라미터는 아니지만 `model.to(device)` 때 모델과 함께 옮겨집니다.
  - `num_classes`(클래스 수)는 기본 2(부정, 긍정)입니다.
  - 크기(`d_model`, `num_heads`, `num_layers`)를 인자로 받습니다.
- **`forward`(계산 순서)**
  1. 토큰 벡터에 위치 인코딩 행렬의 앞 (토큰 수)개 행을 더합니다.
  2. 패딩 마스크(`tokens == 0`)와 함께 인코더에 넣습니다. `nn.TransformerEncoder`에서는 인자 이름이 `src_key_padding_mask`입니다.
  3. 0번 위치의 [CLS] 벡터를 꺼냅니다.
  4. 로짓을 냅니다.

예시 리뷰가 인코더를 지나 [CLS] 벡터가 되는 모습입니다.


<img src="images/34e_sentiment_encoder.png" width="1050" alt="제목은 '인코더: shape은 그대로, [CLS] 위치가 정보를 모읍니다'입니다. 위쪽에 '입력 (1, 64, 64)', '인코더 층 1 어텐션 + FFN', '인코더 층 2 어텐션 + FFN', '출력 (1, 64, 64)' 상자가 화살표로 이어지고 그 아래에 주황 글자 '출력의 0번 위치 = [CLS] 벡터 (1, 64)'가 있습니다. 가운데 토큰 시퀀스는 [CLS], 기, 대, 안, 하, 고, 공백, 봤, 는, ..., [PAD], [PAD]이고, 보라 곡선 화살표 8개가 문자 토큰들에서 [CLS] 카드로 모이며 위에 '[CLS] 위치가 문자 토큰들에서 정보를 모읍니다'가 적혀 있습니다. [PAD] 두 칸에는 빨간 X가 그어져 '[PAD]는 보지 않음', [CLS] 아래에는 '0번 위치', 줄 아래에는 '인코더에 넣는 토큰 시퀀스 (64개 위치 가운데 일부)'가 있습니다. 아래 띠에는 'shape은 그대로 (1, 64, 64), 패딩 마스크로 [PAD] 위치는 보지 않습니다', '[CLS] 벡터는 어텐션으로 리뷰의 문자 토큰들에서 정보를 모은 벡터가 됩니다'가 적혀 있습니다.">


모델을 만들고 파라미터 수를 셉니다.

In [ ]:
# 모델 초깃값을 고정하고, 어휘 크기와 max_len을 넣어 모델을 만듭니다
torch.manual_seed(0)
model = TransformerClassifier(vocab_size=len(vocab), num_classes=2, max_len=max_len).to(device)

# 위치 인코딩 행렬은 버퍼라서 파라미터 수에 들어가지 않습니다
print("파라미터 수:", count_parameters(model))

- **파라미터 수 208,706개:** 토큰 임베딩 행렬(1,697 x 64 = 108,608), 인코더 2층(99,968), 분류 층(64 x 2 + 2 = 130)을 더한 수입니다.

### 10-2. 학습 전 결과

학습 전 모델이 리뷰를 어떻게 판단하는지 먼저 봅니다. 긍정 확률은 로짓 2개에 softmax를 씌운 확률 가운데 클래스 1(긍정)의 값입니다.

In [ ]:
def predict_sentiment(model, texts):
    """리뷰 문장 목록을 받아 긍정 확률(softmax 확률의 클래스 1, 소수 넷째 자리) 목록을 돌려줍니다."""
    model.eval()
    tokens = make_tensor(texts, token_to_id, max_len).to(device)
    with torch.no_grad():
        probabilities = torch.softmax(model(tokens), dim=1)  # (문장 수, 2)
    return [round(p, 4) for p in probabilities[:, 1].tolist()]


# 테스트용 리뷰 6개: 짧은 리뷰, 긴 리뷰, '~지만'이 든 리뷰를 고루 골랐습니다
review_index = [767, 0, 11, 2, 803, 1010]
compare_table = nsmc_test.loc[review_index, ["리뷰", "감성"]].copy()
compare_table["학습 전 긍정 확률"] = predict_sentiment(model, compare_table["리뷰"].tolist())
display(compare_table)

리뷰 몇 개가 아니라 전체 성능도 학습 전 값을 알아 둡니다.

- **평가:** PyTorch 신경망 단원에서 만든 평가 함수 `evaluate`로 검증 데이터의 손실과 정확도를 계산합니다.
- **미니배치:** 학습용은 128개씩 섞어 자르고, 검증용과 테스트용은 순서대로 자릅니다.

In [ ]:
# PyTorch 신경망 단원에서 만든 평가 함수를 그대로 씁니다
def evaluate(model, loader, loss_fn):
    """기울기 없이 예측해 (평균 손실, 정확도)를 돌려줍니다."""
    model.eval()  # 평가 모드: Dropout이 아무것도 0으로 만들지 않습니다
    total_loss = 0.0
    correct = 0
    with torch.no_grad():
        for X_batch, y_batch in loader:
            X_batch = X_batch.to(device)
            y_batch = y_batch.to(device)
            logits = model(X_batch)

            # 마지막 미니배치는 장 수가 적으므로, 손실에 장 수를 곱해 더한 뒤 전체 장 수로 나눕니다
            total_loss += loss_fn(logits, y_batch).item() * len(y_batch)
            correct += (logits.argmax(dim=1) == y_batch).sum().item()

    n_samples = len(loader.dataset)
    return total_loss / n_samples, correct / n_samples

DataLoader 세 개와 손실 함수를 만들고, 학습 전 검증 손실과 정확도를 계산합니다.

In [ ]:
from torch.utils.data import TensorDataset, DataLoader

# 미니배치 128개씩. 학습용만 에폭마다 섞습니다
train_loader = DataLoader(TensorDataset(X_train, y_train), batch_size=128, shuffle=True)
valid_loader = DataLoader(TensorDataset(X_valid, y_valid), batch_size=128, shuffle=False)
test_loader = DataLoader(TensorDataset(X_test, y_test), batch_size=128, shuffle=False)

# 교차 엔트로피 = 평균(-log(정답 클래스의 확률))
loss_fn = nn.CrossEntropyLoss()

# 학습 전 검증 손실과 정확도
valid_loss_before, valid_accuracy_before = evaluate(model, valid_loader, loss_fn)
print("학습 전 검증 손실:", round(valid_loss_before, 4), "| 검증 정확도:", round(valid_accuracy_before, 4))

**결과 읽기**
- **긍정 확률 0.3412~0.4570:** 무작위 파라미터라 리뷰 내용과 상관없이 비슷한 값입니다.
- **검증 정확도 0.4928:** 한쪽으로 답하는 수준입니다.
- **검증 손실 0.7197:** 두 클래스를 같은 확률로 고를 때의 $-\log 0.5 \approx 0.6931$ 근처입니다.

### 10-3. 학습

- **설정:** AdamW(학습률 0.0005, weight decay 0.01), 미니배치 128개, 8에폭입니다.
  - 인코더 층의 Dropout은 기본값 0.1입니다.
  - 미니배치 섞는 순서와 Dropout을 고정하려고 학습 직전에 `torch.manual_seed(0)`을 둡니다.
- **`train_classifier`:** PyTorch 신경망 단원의 학습 반복(`train_one_epoch`, `fit`)에 학습 정확도를 더한 함수입니다.
  - 에폭마다 학습 손실, 학습 정확도, 검증 손실, 검증 정확도를 한 줄로 출력하고 DataFrame으로 남깁니다.
  - **학습 정확도:** 학습 중 미니배치마다 맞힌 수를 더해 계산합니다(Dropout이 켜진 상태의 예측).
- GPU가 없으면 학습에 몇 분 걸릴 수 있습니다.

In [ ]:
def train_classifier(model, train_loader, valid_loader, epochs):
    """AdamW로 epochs번 학습하며, 에폭마다 학습 손실, 학습 정확도, 검증 손실, 검증 정확도를 기록한 DataFrame을 돌려줍니다."""
    optimizer = torch.optim.AdamW(model.parameters(), lr=0.0005, weight_decay=0.01)
    rows = []
    for epoch in range(1, epochs + 1):
        # 학습 모드: Dropout이 켜집니다
        model.train()
        total_loss = 0.0
        correct = 0
        for X_batch, y_batch in train_loader:
            # 미니배치를 모델이 있는 계산 장치로 옮깁니다
            X_batch = X_batch.to(device)  # (배치, 토큰 수)
            y_batch = y_batch.to(device)  # (배치,)

            # 지난 스텝의 기울기를 지우고, 순전파 -> 손실 -> 역전파 -> 갱신
            optimizer.zero_grad()
            logits = model(X_batch)  # (배치, 클래스 수)
            loss = loss_fn(logits, y_batch)
            loss.backward()
            optimizer.step()

            # 이 미니배치의 손실과 맞힌 수를 더합니다
            total_loss += loss.item()
            correct += (logits.argmax(dim=1) == y_batch).sum().item()

        # 학습 손실은 미니배치 손실의 평균, 학습 정확도는 맞힌 수 / 학습 데이터 수
        train_loss = total_loss / len(train_loader)
        train_accuracy = correct / len(train_loader.dataset)

        # 검증 데이터로 평가합니다(evaluate가 평가 모드로 바꿉니다)
        valid_loss, valid_accuracy = evaluate(model, valid_loader, loss_fn)
        print(f"에폭 {epoch}: 학습 손실 {train_loss:.4f}, 학습 정확도 {train_accuracy:.4f} | 검증 손실 {valid_loss:.4f}, 검증 정확도 {valid_accuracy:.4f}")
        rows.append({"에폭": epoch, "학습 손실": train_loss, "학습 정확도": train_accuracy,
                     "검증 손실": valid_loss, "검증 정확도": valid_accuracy})
    return pd.DataFrame(rows).set_index("에폭")


# 학습 전 상태의 모델을 8에폭 학습합니다
epochs = 8
torch.manual_seed(0)
history = train_classifier(model, train_loader, valid_loader, epochs)
display(history.round(4))

에폭마다 기록한 손실과 정확도를 학습 데이터와 검증 데이터로 나눠 그래프 두 개로 봅니다.

In [ ]:
# 왼쪽은 손실, 오른쪽은 정확도. 학습과 검증을 한 그래프에 겹쳐 그립니다
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

axes[0].plot(history.index, history["학습 손실"], marker="o", label="학습")
axes[0].plot(history.index, history["검증 손실"], marker="o", label="검증")
axes[0].set_xlabel("에폭")
axes[0].set_ylabel("손실 (교차 엔트로피)")
axes[0].set_title("영화 리뷰: 손실")
axes[0].legend()

axes[1].plot(history.index, history["학습 정확도"], marker="o", label="학습")
axes[1].plot(history.index, history["검증 정확도"], marker="o", label="검증")
axes[1].set_xlabel("에폭")
axes[1].set_ylabel("정확도")
axes[1].set_title("영화 리뷰: 정확도")
axes[1].legend()

plt.tight_layout()
plt.show()

**결과 읽기**
- **학습 손실:** 에폭마다 줄어드는지 봅니다.
- **검증 정확도:** 학습 전 0.4928에서 얼마나 올랐는지 봅니다. 가장 높은 에폭이 마지막 에폭이 아닐 수도 있습니다.
- **학습과 검증의 차이:** 학습 정확도만 계속 오르고 검증 손실이 다시 오르면 과대적합이 시작된 것입니다.
  - 그 에폭이 PyTorch 신경망 단원에서 본 조기 종료의 멈출 지점입니다.

### 10-4. 학습 전·후 비교

학습이 리뷰 판단을 실제로 바꿨는지 보려고, 학습 전에 본 같은 리뷰 6개와 테스트 데이터로 비교합니다.

In [ ]:
# 같은 리뷰 6개의 학습 전, 학습 후 긍정 확률
compare_table["학습 후 긍정 확률"] = predict_sentiment(model, compare_table["리뷰"].tolist())
display(compare_table)

# 학습을 마친 모델을 테스트 데이터로 한 번 평가합니다
test_loss, test_accuracy = evaluate(model, test_loader, loss_fn)
print("테스트 손실:", round(test_loss, 4), "| 테스트 정확도:", round(test_accuracy, 4))

**결과 읽기**
- **같은 리뷰 6개:** 정답이 긍정(1)인 리뷰는 긍정 확률이 0.5보다 커지고, 부정(0)인 리뷰는 작아졌는지 봅니다.
  - 학습 뒤에도 0.5 근처에 남는 리뷰는 모델이 아직 헷갈리는 리뷰입니다.
- **테스트 정확도:** 처음 보는 리뷰 5,000개에서 맞힌 비율입니다. 학습 전 검증 정확도 0.4928과 비교합니다.

`my_reviews`에 리뷰 문장을 바꿔 넣고 다시 실행해 학습한 모델의 판단을 확인합니다.

In [ ]:
# 리뷰 문장을 바꿔 넣어 보세요
my_reviews = ["배우들 연기가 정말 좋았어요", "시간이 아까운 영화", "기대 안 했는데 생각보다 재밌네요", "스토리는 뻔하지만 영상미는 최고"]
my_table = pd.DataFrame({"리뷰": my_reviews, "긍정 확률": predict_sentiment(model, my_reviews)})
display(my_table)

**결과 읽기**
- 분명한 문장(`시간이 아까운 영화`)과 판단이 섞인 문장(`스토리는 뻔하지만 영상미는 최고`)에서 모델이 어느 쪽으로 판단하는지 봅니다.

## 11. 🖐️ 함께 따라하기: 순위 맞히기 게임

본실습의 인코더를 숫자 게임에 적용합니다. 모델을 만들고 `train_classifier`로 학습해, 인코더가 정답 위치를 고르게 되는지 확인합니다.

**본실습과 다른 점**
- **데이터:** 파일 없이 함수로 만든 숫자 게임입니다. 정답이 규칙으로 정확히 정해집니다.
- **출력:** 클래스 2개(부정, 긍정)의 로짓이 아니라, 숫자 20개의 위치마다 로짓 하나씩, 로짓 20개입니다.

**실습 진행 방법:** `🖐️` 셀의 `...`만 주석에 맞춰 채웁니다. 완성 코드는 `정답/교안_01_정답.ipynb`의 같은 절에 있습니다.

### 11-1. 게임 규칙과 데이터

**게임 규칙**
- **입력 토큰 21개:** 맨 앞 순위 토큰 하나(`|1|`, `|2|`, `|3|`)와, 0~99에서 겹치지 않게 고른 숫자 20개입니다.
- **정답:** k번째로 큰 숫자가 놓인 위치(숫자 20개 가운데 0~19번째)입니다. k는 순위 토큰의 숫자입니다.
  - 정답 위치는 맨 앞 순위 토큰을 빼고 셉니다. 입력 시퀀스의 위치 인덱스보다 1 작습니다.


<img src="images/26_rank_game_board.png" width="1050" alt="제목은 '순위 맞히기 게임: 토큰 21개에서 정답 위치 하나'입니다. 위아래 두 구역이 있습니다. 위 구역 '2번 게임 |3|: 세 번째로 큰 숫자'에는 회색 순위 토큰 |3|과 숫자 카드 20장(아래에 위치 0~19)이 있고, 더 큰 숫자 95(위치 1)와 99(위치 5)는 보라 굵은 테두리, 정답 91(위치 11)은 주황 카드이며 '99 > 95 > 91 -> 정답은 위치 11'이 적혀 있습니다. 아래 구역 '0번 게임 |2|: 두 번째로 큰 숫자'에서는 94(위치 19)가 보라 테두리, 87(위치 13)이 주황이고 '94 > 87 -> 정답은 위치 13'이 적혀 있습니다. 범례는 순위 토큰(회색), 더 큰 숫자(보라 테두리), 정답 숫자(주황)이고, 아래 띠에는 '게임마다 정답 위치가 달라 위치를 외울 수 없습니다', '순위 토큰을 읽고 숫자끼리 비교해야 정답 위치를 찾습니다'가 적혀 있습니다.">


토큰 ID는 숫자 0~99는 그 숫자 그대로, `|1|`, `|2|`, `|3|`은 100, 101, 102입니다. 어휘 크기는 103입니다.

`make_games`는 게임을 만드는 함수입니다. `generator`(난수 생성기)를 받아, 같은 시드면 같은 게임이 나옵니다.

In [ ]:
def make_games(n_games, generator):
    """게임 n_games개를 만들어 (토큰 ID (게임 수, 21), 정답 위치 (게임 수,))를 돌려줍니다."""
    tokens = []
    answer = []
    for _ in range(n_games):
        # 0~99에서 겹치지 않게 숫자 20개를 고르고, 순위 k(1~3)를 고릅니다
        numbers = torch.randperm(100, generator=generator)[:20]
        k = torch.randint(1, 4, (1,), generator=generator).item()

        # 숫자를 큰 순서로 놓았을 때 k번째 숫자의 위치가 정답입니다
        answer.append(numbers.argsort(descending=True)[k - 1].item())

        # [순위 토큰(|k|의 토큰 ID 99 + k), 숫자 20개]
        tokens.append([99 + k] + numbers.tolist())
    return torch.tensor(tokens), torch.tensor(answer)


# 학습 게임 100,000개와 검증 게임 2,000개(서로 다른 시드)
train_games, train_answer = make_games(100000, torch.Generator().manual_seed(1))
valid_games, valid_answer = make_games(2000, torch.Generator().manual_seed(19))
print("학습 게임 shape:", tuple(train_games.shape), "| 검증 게임 shape:", tuple(valid_games.shape))

검증 게임 앞 3개를 봅니다.

In [ ]:
# 앞 3개를 예시 게임으로 씁니다. 순위 토큰의 토큰 ID에서 99를 빼면 k입니다
example_games = valid_games[:3]
example_rows = []
for i in range(3):
    numbers = example_games[i, 1:].tolist()
    example_rows.append({"k": example_games[i, 0].item() - 99,
                         "숫자 20개": " ".join([str(n) for n in numbers]),
                         "정답 위치": valid_answer[i].item(),
                         "정답 숫자": numbers[valid_answer[i]]})
example_table = pd.DataFrame(example_rows)
display(example_table)

**결과 읽기**
- **2번 게임 k = 3:** 99, 95 다음인 91이 11번 위치에 있습니다.
- **0번 게임 k = 2:** 가장 큰 숫자 94 다음인 87이 13번 위치에 있습니다.
- 게임마다 정답 위치가 다르므로, 모델은 위치를 외울 수 없고 숫자를 서로 비교해야 합니다.

### 11-2. 출력과 정확도

- **예측, 손실, 정확도:** 본실습과 같습니다(argmax, 교차 엔트로피 `loss_fn`, `evaluate`).
- **비교 기준:** 아무 위치나 고르면 정확도는 1/20 = 0.05입니다.


<img src="images/27_rank_scores_accuracy.png" width="1050" alt="제목은 '위치마다 로짓 하나: 가장 큰 확률의 위치가 예측 위치'이고, 그 아래에 '2번 게임(|3|)에 맞춘 예시 값'이라고 적혀 있습니다. 왼쪽 막대그래프 '로짓 20개'는 위치 0~19(아래 줄은 그 위치의 숫자)마다 막대가 하나씩 있고 위치 11(숫자 91) 막대만 주황으로 가장 높습니다. 가운데 'softmax' 화살표를 지나 오른쪽 막대그래프 'softmax 확률 20개 (합 1)'에서는 위치 11 막대가 주황으로 가장 높고 '예측 위치 11 (확률 0.58)' 화살표가 붙어 있습니다. 아래 띠에는 '리뷰 분류: 클래스 2개 (부정, 긍정) | 순위 게임: 출력 20개 (위치 0~19)', 'softmax, 교차 엔트로피, argmax는 리뷰 분류와 같습니다', '정확도 = 예측 위치가 정답 위치와 같은 게임 수 / 전체 게임 수, 아무 위치나 고르면 1/20 = 0.05'가 적혀 있습니다.">


### 11-3. 모델: RankClassifier

`TransformerClassifier`와 같은 부품(토큰 임베딩, 위치 인코딩, `nn.TransformerEncoder`)에 출력 층만 다르게 붙인 모델입니다.

In [ ]:
from encoder_models import RankClassifier

# 모델 코드를 그대로 출력합니다
print(inspect.getsource(RankClassifier))

**`TransformerClassifier`와 달라진 점**
- **출력 층 `score = nn.Linear(64, 1)`:** [CLS] 벡터 하나 대신, 맨 앞 순위 토큰을 뺀 숫자 20개 위치 `h[:, 1:]`의 벡터마다 로짓 하나를 냅니다. 로짓 shape은 (배치, 20)입니다.
- **패딩 마스크 없음:** 모든 게임이 토큰 21개라 [PAD]가 없습니다.

### 11-4. 학습 전·후 비교

DataLoader 두 개와 모델을 만들고, 학습 전 검증 정확도와 예시 게임 3개의 예측 위치를 먼저 봅니다.

In [ ]:
# 🖐️ 함께 따라하기: 학습 게임은 256개씩 섞어서, 검증 게임은 256개씩 순서대로 자르는 DataLoader를 만드세요
rank_train_loader = ...
rank_valid_loader = ...

# 🖐️ 함께 따라하기: RankClassifier를 기본값 그대로 만들어 계산 장치에 올리세요
torch.manual_seed(0)
rank_model = ...

# 학습 전 검증 손실과 정확도
rank_loss_before, rank_accuracy_before = evaluate(rank_model, rank_valid_loader, loss_fn)
print("학습 전 검증 손실:", round(rank_loss_before, 4), "| 검증 정확도:", round(rank_accuracy_before, 4))

# 예시 게임 3개의 학습 전 예측 위치
with torch.no_grad():
    example_table["학습 전 예측 위치"] = rank_model(example_games.to(device)).argmax(dim=1).cpu().tolist()
display(example_table)

**결과 읽기**
- **학습 전 검증 정확도 0.102:** 아직 무작위 파라미터라 낮습니다.

10-3의 `train_classifier`로 10에폭 학습합니다. 에폭마다 학습과 검증의 손실, 정확도가 한 줄씩 출력됩니다.

In [ ]:
# 🖐️ 함께 따라하기: train_classifier로 rank_model을 10에폭 학습해 기록을 rank_history에 받으세요
torch.manual_seed(0)
rank_history = ...
display(rank_history.round(4))

예시 게임 3개의 학습 후 예측 위치를 학습 전과 나란히 봅니다.

In [ ]:
# 🖐️ 함께 따라하기: 예시 게임 3개의 학습 후 예측 위치를 example_table에 더하세요
with torch.no_grad():
    example_table["학습 후 예측 위치"] = ...
display(example_table)

에폭마다 기록한 손실과 정확도를 그래프로 봅니다.

In [ ]:
# 왼쪽은 손실, 오른쪽은 정확도. 학습과 검증을 한 그래프에 겹쳐 그립니다
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

axes[0].plot(rank_history.index, rank_history["학습 손실"], marker="o", label="학습")
axes[0].plot(rank_history.index, rank_history["검증 손실"], marker="o", label="검증")
axes[0].set_xlabel("에폭")
axes[0].set_ylabel("손실 (교차 엔트로피)")
axes[0].set_title("순위 맞히기 게임: 손실")
axes[0].legend()

axes[1].plot(rank_history.index, rank_history["학습 정확도"], marker="o", label="학습")
axes[1].plot(rank_history.index, rank_history["검증 정확도"], marker="o", label="검증")
axes[1].axhline(0.05, color="gray", linestyle="--", label="아무 위치나 고르기 (0.05)")
axes[1].set_xlabel("에폭")
axes[1].set_ylabel("정확도")
axes[1].set_title("순위 맞히기 게임: 정확도")
axes[1].legend()

plt.tight_layout()
plt.show()

**결과 읽기**
- **학습 후 검증 정확도:** 0.05보다 훨씬 높아야 합니다. 인코더가 순위 토큰과 숫자끼리 비교해 정답 위치를 고르게 된 것입니다.
- **예시 게임 3개:** 학습 후 예측 위치가 정답 위치(13, 9, 11)와 같은지 봅니다.

## 12. 핵심 정리

- **트랜스포머:** 모든 토큰이 다른 모든 토큰을 한꺼번에 보는 어텐션 블록을 쌓은 모델입니다.
  - 인코더는 입력 전체를 읽어 토큰마다 벡터를 내고, 디코더는 causal 마스크로 앞 토큰만 보며 다음 토큰을 냅니다.
  - 세 갈래는 Encoder-only(BERT), Decoder-only(GPT·Llama·Qwen), Encoder-Decoder(원 논문·T5)입니다.
- **토큰화와 어휘 사전:** 문자 단위로 토큰화하고, 학습 데이터로만 만든 어휘 사전([PAD] 0, [UNK] 1, [CLS] 2)에서 토큰 ID를 찾습니다.
  - [CLS]를 붙이고 `max_len` 64로 자르거나 [PAD]로 채웁니다.
- **토큰 임베딩:** `nn.Embedding(어휘 크기, d_model)`은 토큰 ID로 임베딩 행렬(학습되는 파라미터)의 행을 꺼냅니다. 같은 토큰 ID는 늘 같은 벡터라 문맥이 없습니다.
- **어텐션:** Query와 모든 Key의 어텐션 점수를 $\sqrt{d_k}$로 나눠 softmax한 어텐션 가중치(합 1)로 Value를 가중합합니다.
  - 어텐션 가중치를 토큰의 내용으로 정합니다. query, key, value가 같은 토큰 벡터면 Self-Attention입니다.
  - 토큰 순서를 바꿔 넣으면 출력의 순서만 바뀝니다. 어텐션은 순서를 모릅니다.
- **위치 인코딩:** 위치마다 sin, cos로 계산한 고정 벡터를 토큰 벡터에 더해 순서를 알려 줍니다. 학습하지 않습니다.
- **Multi-Head Attention과 패딩 마스크:** `nn.MultiheadAttention(embed_dim, num_heads, batch_first=True)`는 d_model을 헤드 수로 나눠 헤드마다 따로 어텐션 가중치를 계산합니다.
  - [PAD] 위치가 True인 마스크를 `key_padding_mask`로 넘기면 [PAD] 위치에 어텐션 가중치를 주지 않습니다.
- **인코더 블록:** [Multi-Head Self-Attention → 더하기 + LayerNorm → FFN → 더하기 + LayerNorm]입니다.
  - `nn.TransformerEncoderLayer` 하나가 블록 하나(층 하나 49,984개), `nn.TransformerEncoder`가 쌓은 인코더(2층 99,968개)입니다.
  - LayerNorm은 토큰 하나 안에서, BatchNorm은 특징 하나를 배치 전체에서 평균과 분산을 계산합니다. 문장에는 다른 문장과 [PAD]의 영향을 받지 않는 LayerNorm을 씁니다.
- **감성 분석:** [CLS] 벡터를 `nn.Linear`로 분류합니다. 에폭마다 학습과 검증의 손실, 정확도를 함께 보고 과대적합을 살핍니다.